In [ ]:

# Stress Controlled Loading


import os, sys, csv          #IMporting basic python libraries
import numpy as np

# import pyexadis
pyexadis_paths = [
    '../../python',
    '../../lib',
    '../../core/pydis/python',        # possible paths where OpenDiS Python modules may exist.
    '../../core/exadis/python/'
]

[
    sys.path.append(os.path.abspath(p))
    for p in pyexadis_paths
    if p not in sys.path
]

import pyexadis        # importing main pyexadis module
from pyexadis_base import ExaDisNet, DisNetManager
from pyexadis_base import CalForce, MobilityLaw, TimeIntegration
from pyexadis_base import Collision, Topology, Remesh, SimulateNetwork


#  helper functions (small utility functions used throughout the simulation)

def voigt_sym_to_tensor(v6):
    #  converts Voigt order: [xx, yy, zz, yz, zx, xy] into 3×3 symmetric tensor.
    v6 = np.asarray(v6, float).ravel()

    return np.array([
        [v6[0], v6[5], v6[4]],
        [v6[5], v6[1], v6[3]],
        [v6[4], v6[3], v6[2]]
    ], float)


def von_mises_stress(S):                     # Calculating von Mises stress from a 3×3 stress tensor.
    S = np.array(S, float).reshape(3, 3)
    Sdev = S - np.trace(S)/3.0*np.eye(3)

    return np.sqrt(1.5*np.tensordot(Sdev, Sdev, 2))


def von_mises_eq_strain(E):                  # Calculating equivalent strain
    E = np.array(E, float).reshape(3, 3)
    Edev = E - np.trace(E)/3.0*np.eye(3)

    return np.sqrt((2.0/3.0)*np.tensordot(Edev, Edev, 2))


def Eeq(LA, MU):                              # Calculating Young’s modulus E from Lambda & shear modulus
    return MU*(3*LA + 2*MU)/(LA + MU)


def nu(LA, MU):
    return 0.5*LA/(LA + MU)                   # Calculating Poissons Ratio


class StopSimulation(RuntimeError):
    # Used to stop the simulation once we reach taregt stress and not max_step
    pass


class MySimulateNetworkPerf(SimulateNetwork):  # We are using default OpenDiS simulationdriver but customizing it

    def __init__(
        self,
        *args,
        LA,
        MU,
        stress_rate_tensor=None,              # Initializing the simulation object
        # stop controls
        t_total=None,
        target_sigma_zz=None,
        tol_time=0.0,
        tol_sigma=0.0,
        **kwargs
    ):

        super().__init__(*args, **kwargs)

        self.LA = float(LA)                    # Storing lambda and mu
        self.MU = float(MU)

        self._Sdot_voigt = np.asarray(
            stress_rate_tensor
            if stress_rate_tensor is not None    # storing stress rate tensor in Voigt format.
            else np.zeros(6),
            float
        )

        self.t_total = None if t_total is None else float(t_total)  # Storing total simulation time
        self.target_sigma_zz = (
            None if target_sigma_zz is None
            else float(target_sigma_zz)             # Storing the target axial stress.
        )

        self.tol_time = float(tol_time)
        self.tol_sigma = float(tol_sigma)          # tolerance values so that the simulation can terminate as it is very close to target

        # robust user-time accumulator
        self._time_cum = 0.0

        # output paths and storing the output folders
        self._out_dir = self.write_dir

        self._out_path = os.path.join(
            self._out_dir,
            "stress_strain_dens.dat"                # Creates path for output files
        )

        self._csv_main = os.path.join(
            self._out_dir,
            "stress_strain_dens_main.csv"
        )                                           # Here we have main csv and a power csv file where power file only has thermodynamic check

        self._csv_power = os.path.join(
            self._out_dir,
            "stress_strain_dens_power.csv"
        )

        self._wrote_header_dat = os.path.exists(self._out_path)
        self._wrote_header_csv = os.path.exists(self._csv_main)
        self._wrote_header_pow = os.path.exists(self._csv_power)

        self._step_fallback = 0
        self._ep_eq_cum = 0.0     # stores accumulated equivalent plastic strain:

    def step_update_response(self, N: DisNetManager, state: dict):      # Main update function which we call every step
                                                                        #  N is dislocation network manager.
        if self.loading_mode == 'stress_rate_tensor':

            dEp_T, dWp_vec, density = (
                N.get_disnet(ExaDisNet)
                 .net
                 .get_plastic_strain()
            )

            state["density"] = float(density)                           # Storing Density

            # extracting plastic deformation from the dislocation network.
            dEp = np.array(dEp_T, float).ravel()[[0, 4, 8, 5, 2, 1]]
                                                                           # The ordering is as per the convention in library
            dWp = np.array(dWp_vec, float).ravel()[[5, 2, 1]]
            # (Plastic Spin)
            state["dEp"] = dEp
            state["dWp"] = dWp          # plastic strain increment and plastic spin increment saved

            dt = float(state["dt"])

            # clamping final increment
            if self.t_total is not None:

                rem = self.t_total - self._time_cum

                if rem <= 0.0:
                    dt_eff = 0.0
                else:
                    dt_eff = min(dt, rem)

            else:
                dt_eff = dt

            # advance our own time
            self._time_cum += dt_eff

            # apply stress increment
            dS_voigt = self._Sdot_voigt * dt_eff      # Computing stress increment

            state["applied_stress"] = (
                state.get("applied_stress", np.zeros(6))    # Updating applied stress
                + dS_voigt
            )

            # compliance matrix formation
            E_ = Eeq(self.LA, self.MU)
            nu_ = nu(self.LA, self.MU)

            S = np.zeros((6, 6), float)

            for i in range(3):
                for j in range(3):
                    S[i, j] = -nu_/E_

                S[i, i] = 1.0/E_

            for i in range(3, 6):
                S[i, i] = 1.0/(2.0*self.MU)

            dEe = S @ dS_voigt          # Elastic Strain increment
            dE_voigt = dEe + dEp        # Total Strain increment


            state["Etot"] = (
                state.get("Etot", np.zeros(6))      # Accumulated total strain
                + dE_voigt
            )

            state["stress"] = float(
                von_mises_stress(
                    voigt_sym_to_tensor(state["applied_stress"])
                )
            )

            state["strain"] = float(
                von_mises_eq_strain(
                    voigt_sym_to_tensor(state["Etot"])
                )
            )

        else:

            super().step_update_response(N, state)

            self._time_cum += float(state.get("dt", 0.0))

        return state

    #function is called at the end of each simulation step.
    def step_end(self, N: DisNetManager, state: dict):

        super().step_end(N, state)

        t = float(self._time_cum)

        sig = np.asarray(
            state.get("applied_stress", np.zeros(6)),
            float
        ).ravel()

        sigma_zz = float(sig[2]) if sig.size >= 3 else 0.0

        time_hit = (
            (self.t_total is not None)
            and
            (t >= self.t_total - self.tol_time)
        )

        sigma_hit = (
            (self.target_sigma_zz is not None)
            and
            (sigma_zz >= self.target_sigma_zz - self.tol_sigma)
        )

        if time_hit or sigma_hit:

            raise StopSimulation(
                f"Stop: time={t:.6e}s, sigma_zz={sigma_zz:.6e}Pa"
            )
    # Output WRiting Function
    def step_write_files(self, N: DisNetManager, state: dict):

        step = int(
            state.get(
                "istep",
                state.get("step", self._step_fallback + 1)
            )
        )

        if self.write_freq and (                    # Controlling the output frequency
            step % int(self.write_freq) != 0
        ):
            return

        os.makedirs(self._out_dir, exist_ok=True)

        if not self._wrote_header_dat:

            with open(self._out_path, "w") as f:

                f.write(
                    "# step time(s) dt(s) strain_eq sigma_vm(Pa) "
                    "s_xx(Pa) s_yy(Pa) s_zz(Pa) "
                    "s_yz(Pa) s_zx(Pa) s_xy(Pa) "
                    "density(1/m^2) "                        # File headers
                    "Lpxx Lpxy Lpxz Lpyx Lpyy "
                    "Lpyz Lpzx Lpzy Lpzz "
                    "epdot_eq(1/s) ep_eq(-)\n"
                )

            self._wrote_header_dat = True

        if not self._wrote_header_csv:

            with open(self._csv_main, "w", newline="") as fcsv:

                w = csv.writer(fcsv)

                w.writerow([
                    "step",
                    "time(s)",
                    "dt(s)",
                    "strain_eq",
                    "sigma_vm(Pa)",
                    "s_xx(Pa)",
                    "s_yy(Pa)",
                    "s_zz(Pa)",
                    "s_yz(Pa)",
                    "s_zx(Pa)",
                    "s_xy(Pa)",
                    "density(1/m^2)",
                    "Lpxx",
                    "Lpxy",
                    "Lpxz",
                    "Lpyx",
                    "Lpyy",
                    "Lpyz",
                    "Lpzx",
                    "Lpzy",
                    "Lpzz",
                    "epdot_eq(1/s)",
                    "ep_eq(-)"
                ])

            self._wrote_header_csv = True

        if not self._wrote_header_pow:

            with open(self._csv_power, "w", newline="") as fcsv:

                w = csv.writer(fcsv)

                w.writerow([
                    "step",
                    "p_plast(W/m^3)",
                    "p_spin(W/m^3)",
                    "p_full(W/m^3)"
                ])

            self._wrote_header_pow = True

        dEp = voigt_sym_to_tensor(state["dEp"])  # converting plastic strain increment from Voigt to tensor:

        wyz, wxz, wxy = state["dWp"]

        dWp = np.array([
            [0.0,  wxy,  wxz],
            [-wxy, 0.0,  wyz],
            [-wxz, -wyz, 0.0]
        ], float)

        dt = float(state["dt"])
        time = float(self._time_cum)

        Lp = (dEp + dWp) / max(dt, 1e-30)   # Constructing Lp and Dp and 1e-40 is used for avoiding division by zero, a code choice
        Dp = dEp / max(dt, 1e-30)

        epdot_eq = float(
            np.sqrt(
                (2.0/3.0)
                *
                np.tensordot(Dp, Dp, axes=2)
            )
        )

        self._ep_eq_cum += epdot_eq * dt

        sigT = voigt_sym_to_tensor(state["applied_stress"])

        p_plast = float(np.tensordot(sigT, Dp, axes=2))

        p_spin = float(
            np.tensordot(
                sigT,
                dWp/max(dt, 1e-30),
                axes=2
            )
        )

        p_full = p_plast + p_spin

        strain_scalar = float(state.get("strain", 0.0))
        sigma_vm = float(state.get("stress", 0.0))
        density_scalar = float(state.get("density", 0.0))

        v6 = np.asarray(
            state["applied_stress"],
            float
        ).ravel()

        # Writing output rows at each step
        with open(self._out_path, "a") as f:

            f.write(
                f"{step:8d} "
                f"{time: .8e} "
                f"{dt: .8e} "
                f"{strain_scalar: .8e} "
                f"{sigma_vm: .8e} "
                +
                " ".join(f"{x: .8e}" for x in v6)
                +
                " "
                +
                f"{density_scalar: .8e} "
                +
                " ".join(
                    f"{x: .6e}"
                    for x in Lp.ravel(order='C')
                )
                +
                " "
                +
                f"{epdot_eq: .6e} "
                f"{self._ep_eq_cum: .6e}\n"
            )

            f.write(
                f" {p_plast: .6e} "
                f"{p_spin: .3e} "
                f"{p_full: .6e}\n"
            )

        with open(self._csv_main, "a", newline="") as fcsv:  # for main csv file

            w = csv.writer(fcsv)

            row = (
                [step, time, dt, strain_scalar, sigma_vm]
                +
                list(v6)
                +
                [density_scalar]
                +
                list(Lp.ravel(order='C'))
                +
                [epdot_eq, self._ep_eq_cum]
            )

            w.writerow(row)

        with open(self._csv_power, "a", newline="") as fcsv:  # For power csv file

            w = csv.writer(fcsv)

            w.writerow([
                step,
                p_plast,
                p_spin,
                p_full
            ])


def run_stress_controlled():       # This function sets up and runs the full DDD simulation.

    pyexadis.initialize()

    try:

        state = {
            "crystal": 'fcc',
            "burgmag": 2.55e-10,
            "mu": 54.6e9,
            "nu": 0.324,
            "a": 6.0,                          # Do not change until you see max_conn error and keep this constant
            "maxseg": 5000.0,
            "minseg": 1200.0,
            "rtol": 13.0,
            "rann": 9.0,
            "nextdt": 2e-11,
            "maxdt": 1.5e-10,
        }

        MU = float(state["mu"])
        nu_ = float(state["nu"])

        LA = 2.0*MU*nu_/(1.0 - 2.0*nu_)

        Lbox = 58824.0

        G = ExaDisNet().generate_line_config(
            state["crystal"],
            Lbox,
            num_lines=15*12,   # Change this for dislocation density
            theta=[0.0, 60.0, 90.0],
            maxseg=state["maxseg"],
            seed=1234
        )

        net = DisNetManager(G)

        calforce = CalForce(
            force_mode='SUBCYCLING_MODEL',
            state=state,                           # Force calculator
            Ngrid=64,
            cell=net.cell
        )

        mobility = MobilityLaw(
            mobility_law='FCC_0',
            state=state,                            # Velocity update
            Medge=64103.0,
            Mscrew=64103.0,
            vmax=2000.0
        )

        timeint = TimeIntegration(
            integrator='Subcycling',                 # Time integrator which updates position
            rgroups=[0.0, 80.0, 400.0, 1400.0],
            state=state,
            force=calforce,
            mobility=mobility
        )

        collision = Collision(
            collision_mode='Retroactive',
            state=state
        )

        topology = Topology(
            topology_mode='TopologyParallel',
            state=state,
            force=calforce,
            mobility=mobility
        )

        remesh = Remesh(
            remesh_rule='LengthBased',
            state=state
        )

        # RATE CONTROL INPUTS

        target_sigma = 50e6  # Change this for different sresses
        sigma_dot = 6e13     # Change this for rate change

        t_ramp = target_sigma / sigma_dot

        stress_rate_tensor = np.array(
            [0., 0., sigma_dot, 0., 0., 0.],
            float
        )

        write_dir = "50Mpa_6e13"      # Change this for every new simulation

        sim = MySimulateNetworkPerf(
            calforce=calforce,
            mobility=mobility,
            timeint=timeint,
            collision=collision,
            topology=topology,
            remesh=remesh,
            vis=None,
            loading_mode='stress_rate_tensor',
            burgmag=state["burgmag"],
            state=state,
            print_freq=100,
            plot_freq=0,
            write_freq=1,
            write_dir=write_dir,
            LA=LA,
            MU=MU,
            stress_rate_tensor=stress_rate_tensor,
            t_total=t_ramp,
            target_sigma_zz=target_sigma,
            tol_time=0.0,
            tol_sigma=0.0,
            max_step=100000   # JUst a cap
        )

        sim.run(net, state)

    except StopSimulation as e:

        print(str(e))

    finally:

        pyexadis.finalize()


if __name__ == "__main__":
    run_stress_controlled()

In [ ]:
#Strain Controlled Loading for uniaxial loading
import os, sys, csv
import numpy as np           # Importing basic python libraries

# importing pyexadis

pyexadis_paths = ['../../python', '../../lib',
                  '../../core/pydis/python',
                  '../../core/exadis/python/']   # possible paths where OpenDiS Python modules may exist.
for p in pyexadis_paths:
    ap = os.path.abspath(p)
    if ap not in sys.path:
        sys.path.append(ap)

import pyexadis               # importing main pyexadis module
from pyexadis_base import ExaDisNet, DisNetManager
from pyexadis_base import CalForce, MobilityLaw, TimeIntegration
from pyexadis_base import Collision, Topology, Remesh, SimulateNetwork

# helpers (small utility functions used throughout the simulation)

def voigt_sym_to_tensor(v6):     # Converting a 6-component Voigt vector into a 3x3 symmetric tensor.

    # Voigt order used: [xx, yy, zz, yz, zx, xy]
    v6 = np.asarray(v6, float).ravel()
    return np.array([[v6[0], v6[5], v6[4]],
                     [v6[5], v6[1], v6[3]],
                     [v6[4], v6[3], v6[2]]], float)

def tensor_to_voigt_sym(T):      # Converting a 3x3 symmetric tensor into 6-component Voigt form.
    # returns [xx, yy, zz, yz, zx, xy]
    T = np.asarray(T, float).reshape(3, 3)
    return np.array([T[0,0], T[1,1], T[2,2], T[1,2], T[2,0], T[0,1]], float)

def von_mises_stress(S):
    S = np.array(S, float).reshape(3, 3)
    Sdev = S - np.trace(S)/3.0*np.eye(3)               # Calculating Von mises stress removing hydrostatic stress
    return np.sqrt(1.5*np.tensordot(Sdev, Sdev, 2))

def von_mises_eq_strain(E):
    E = np.array(E, float).reshape(3, 3)               # Calculating equivalent strain
    Edev = E - np.trace(E)/3.0*np.eye(3)
    return np.sqrt((2.0/3.0)*np.tensordot(Edev, Edev, 2))

def lame_lambda(mu, nu):
    return 2.0*mu*nu/(1.0 - 2.0*nu)       # Calculating lambda from shear modulus and Poisson ratio.

def build_C_voigt(LA, MU):

    C = np.zeros((6,6), float)
    # normal
    for i in range(3):
        for j in range(3):               # Building isotropic stiffness matrix C in Voigt form.
            C[i,j] = LA
        C[i,i] += 2.0*MU
    # shear
    for i in range(3,6):
        C[i,i] = 2.0*MU
    return C

# Clean stopping condition
class StopSimulation(RuntimeError):
    # Custom exception used to stop the simulation cleanly once we reach traget strain
    pass

class MyStrainRateSimulateNetwork(SimulateNetwork):
    """
    Strain-rate controlled driver with:
    - full sigma(6) update via isotropic elasticity: dσ = C:(dE - dEp)
    - full Lp(9) output from (dEp + dWp)/dt_eff
    - CSV writing gated by write_freq (so write_freq=1 logs every step)
    """
    def __init__(self, *args, LA, MU,
                 erate, edir,
                 max_strain=None,       # Acting as Constructor for the strain-rate controlled simulation.

                 tol_strain=0.0,
                 **kwargs):
        super().__init__(*args, **kwargs)
        self.LA, self.MU = float(LA), float(MU)       # storing elastic constants
        self.C = build_C_voigt(self.LA, self.MU)
        self.erate = float(erate)                     #  Storing imposed strain rate
        edir = np.asarray(edir, float).ravel()        #  Converting loading direction into NumPy array
        if edir.size != 3:
            raise ValueError("edir must be 3-vector")
        nrm = np.linalg.norm(edir)
        if nrm <= 0.0:                                 # Conditions on edir
            raise ValueError("edir must be nonzero")
        self.edir = edir / nrm

        self.max_strain = None if max_strain is None else float(max_strain) # Storing final target strain and tolerance
        self.tol_strain = float(tol_strain)

        # user defined time accumulator
        self._time_cum = 0.0

        # Accumulated imposed scalar strain
        self._eps_imposed = 0.0

        # cumulative equivalent plastic strain
        self._ep_eq_cum = 0.0

        # output files
        self._out_dir   = self.write_dir
        self._csv_main  = os.path.join(self._out_dir, "stress_strain_dens_main.csv")

        self._wrote_header_csv = os.path.exists(self._csv_main)
        self._step_fallback = 0

    def step_update_response(self, N: DisNetManager, state: dict):  # Called at every step and performs update
        if self.loading_mode == 'strain_rate':
            # plastic increments from ExaDiS
            dEp_T, dWp_vec, density = N.get_disnet(ExaDisNet).net.get_plastic_strain()   # get_plastic_strain returns plastic strain and spin increment
            state["density"] = float(density)

            # dEp_T is 3x3 flattened; reorder to [xx,yy,zz,yz,xz,xy]
            dEp = np.array(dEp_T, float).ravel()[[0, 4, 8, 5, 2, 1]]

            dWp = np.array(dWp_vec, float).ravel()[[5, 2, 1]]

            state["dEp"], state["dWp"] = dEp, dWp

            dt = float(state["dt"])    # Opendis adaptive time step

            # clamp eddt so we land exactly at max_strain (imposed scalar)
            dt_eff = dt
            if self.max_strain is not None:
                rem_eps = self.max_strain - self._eps_imposed
                if rem_eps <= 0.0:
                    dt_eff = 0.0
                else:
                    dt_eff = min(dt, rem_eps / max(self.erate, 1e-30))

            # Updating accumulated time and imposed strain
            self._time_cum += dt_eff
            self._eps_imposed += self.erate * dt_eff

            state["time"] = self._time_cum
            state["dt_eff"] = dt_eff
            state["eps_imposed"] = self._eps_imposed

            # building imposed total strain increment tensor: dE = erate*dt_eff*(n⊗n)
            dE_tensor = (self.erate * dt_eff) * np.outer(self.edir, self.edir)
            dE_voigt = tensor_to_voigt_sym(dE_tensor)

            # elastic increment in Voigt: dEe = dE - dEp
            dEe = dE_voigt - dEp

            # stress increment: dS = C : dEe
            dS_voigt = self.C @ dEe

            # accumulate total strain and stress
            state["Etot"] = state.get("Etot", np.zeros(6)) + dE_voigt
            state["applied_stress"] = state.get("applied_stress", np.zeros(6)) + dS_voigt

            # convenient scalars (matching stress-controlled style)
            state["stress"] = float(von_mises_stress(voigt_sym_to_tensor(state["applied_stress"])))
            state["strain"] = float(von_mises_eq_strain(voigt_sym_to_tensor(state["Etot"])))
        else:
            super().step_update_response(N, state)
            self._time_cum += float(state.get("dt", 0.0))
            state["time"] = self._time_cum

        return state

    def step_end(self, N: DisNetManager, state: dict):   #  Called at the end of each time step
        super().step_end(N, state)

        # stop when imposed scalar strain reaches target
        if self.loading_mode == 'strain_rate' and self.max_strain is not None:
            if self._eps_imposed >= self.max_strain - self.tol_strain:
                raise StopSimulation(f"Stop: eps_imposed={self._eps_imposed:.6e}, time={self._time_cum:.6e}s")

    def step_write_files(self, N: DisNetManager, state: dict):   # Writes simulation output to CSV file
        step = int(state.get("istep", state.get("step", self._step_fallback + 1)))
        self._step_fallback = step

        if self.write_freq and (step % int(self.write_freq) != 0):
            return

        os.makedirs(self._out_dir, exist_ok=True)

        if not self._wrote_header_csv:
            with open(self._csv_main, "w", newline="") as fcsv:
                w = csv.writer(fcsv)
                w.writerow(["step","time(s)","dt(s)","dt_eff(s)","eps_imposed",
                            "strain_eq","sigma_vm(Pa)",
                            "s_xx(Pa)","s_yy(Pa)","s_zz(Pa)","s_yz(Pa)","s_zx(Pa)","s_xy(Pa)",
                            "density(1/m^2)",
                            "Lpxx","Lpxy","Lpxz","Lpyx","Lpyy","Lpyz","Lpzx","Lpzy","Lpzz",
                            "epdot_eq(1/s)","ep_eq(-)"])
            self._wrote_header_csv = True

        # fetching state
        v6 = np.asarray(state.get("applied_stress", np.zeros(6)), float).ravel()
        Etot6 = np.asarray(state.get("Etot", np.zeros(6)), float).ravel()

        dt = float(state.get("dt", 0.0))
        dt_eff = float(state.get("dt_eff", dt))
        time = float(state.get("time", 0.0))
        eps_imposed = float(state.get("eps_imposed", 0.0))

        strain_scalar  = float(state.get("strain", von_mises_eq_strain(voigt_sym_to_tensor(Etot6))))
        sigma_vm       = float(state.get("stress", von_mises_stress(voigt_sym_to_tensor(v6))))
        density_scalar = float(state.get("density", 0.0))

        # Lp from plastic increments
        dEp = voigt_sym_to_tensor(state["dEp"])   # Reconstructing plastic strain and plastic spin tensors
        wyz, wxz, wxy = state["dWp"]
        dWp = np.array([[ 0.0,  wxy,  wxz],
                        [-wxy,  0.0,  wyz],
                        [-wxz, -wyz,  0.0]], float)

        denom = max(dt_eff, 1e-30)
        Lp = (dEp + dWp) / denom
        Dp = dEp / denom

        epdot_eq = float(np.sqrt((2.0/3.0) * np.tensordot(Dp, Dp, axes=2)))
        self._ep_eq_cum += epdot_eq * dt_eff

        with open(self._csv_main, "a", newline="") as fcsv:
            w = csv.writer(fcsv)
            row = [step, time, dt, dt_eff, eps_imposed,
                   strain_scalar, sigma_vm] + list(v6) + [density_scalar] \
                  + list(Lp.ravel(order='C')) + [epdot_eq, self._ep_eq_cum]
            w.writerow(row)

    def step_print_info(self, N: DisNetManager, state: dict):  # Prints useful information to terminal during simulation controlled by print_freq
        super().step_print_info(N, state)

        step = int(state.get("istep", 0))
        if self.print_freq and (step % int(self.print_freq) != 0):
            return state

        v6 = np.asarray(state.get("applied_stress", np.zeros(6)), float).ravel()
        print("\n--- STRAIN-RATE FULL OUTPUT ---")
        print(f"step={step}  time={float(state.get('time',0.0)):.6e}  dt={float(state.get('dt',0.0)):.3e}  dt_eff={float(state.get('dt_eff',0.0)):.3e}")
        print(f"eps_imposed={float(state.get('eps_imposed',0.0)):.6e}  strain_eq={float(state.get('strain',0.0)):.6e}  sigma_vm={float(state.get('stress',0.0)):.6e}")
        print("sigma6 [xx yy zz yz zx xy] (Pa) =", v6)
        return state

def run_strain_controlled():   # Main function setting and running the strain-controlled DDD simulation
    pyexadis.initialize()
    try:
        state = {
            "crystal": 'fcc',
            "burgmag": 2.55e-10,
            "mu": 54.6e9,
            "nu": 0.324,
            "a": 6.0,
            "maxseg": 5000.0,           # Keep these same until max_conn error is encountered
            "minseg": 1200.0,
            "rtol": 12.0,
            "rann": 8.5,
            "nextdt": 2e-11,
            "maxdt": 1.5e-10,
        }

        MU = float(state["mu"])
        nu_ = float(state["nu"])
        LA = lame_lambda(MU, nu_)

        # microstructure details
        Lbox = 58824.0
        G = ExaDisNet().generate_line_config(
            state["crystal"], Lbox,
            num_lines=15*12,    # change this only for studying density effects
            theta=[0.0, 60.0, 90.0],
            maxseg=state["maxseg"],
            seed=1234
        )
        net = DisNetManager(G)

        # physics modules
        calforce = CalForce(force_mode='SUBCYCLING_MODEL',
                            state=state, Ngrid=96, cell=net.cell)
        mobility = MobilityLaw(mobility_law='FCC_0', state=state,
                               Medge=64103.0, Mscrew=64103.0, vmax=2000.0)
        timeint = TimeIntegration(integrator='Subcycling',
                                  rgroups=[0.0, 120.0, 400.0, 1400.0],
                                  state=state, force=calforce,
                                  mobility=mobility)
        collision = Collision(collision_mode='Retroactive', state=state)
        topology  = Topology(topology_mode='TopologyParallel',
                             state=state, force=calforce,
                             mobility=mobility)
        remesh    = Remesh(remesh_rule='LengthBased', state=state)

        # strain-rate loading inputs
        erate = 1e4   # Change for different strain rates
        edir = np.array([0.0, 0.0, 1.0], float) # Change for loading direction
        max_strain = 0.005

        write_dir = "straincontrol1"  # Change always for new simulation

        sim = MyStrainRateSimulateNetwork(
            calforce=calforce, mobility=mobility, timeint=timeint,
            collision=collision, topology=topology, remesh=remesh,
            vis=None,
            loading_mode='strain_rate',
            burgmag=state["burgmag"], state=state,
            write_freq=1,
            print_freq=50,
            plot_freq=0,

            write_dir=write_dir,
            LA=LA, MU=MU,
            erate=erate, edir=edir,
            max_strain=max_strain,
            tol_strain=0.0,
            max_step=2000000
        )

        sim.run(net, state)

    except StopSimulation as e:
        print(str(e))
    finally:
        pyexadis.finalize()

if __name__ == "__main__":
    run_strain_controlled()

In [ ]:
import os
import sys
import csv
import numpy as np

# Import pyexadis / OpenDiS (cyclicreducedcomp.py)
pyexadis_paths = [
    '../../python',
    '../../lib',
    '../../core/pydis/python',       # possible paths where OpenDiS Python modules may exist.
    '../../core/exadis/python/'
]
[sys.path.append(os.path.abspath(path)) for path in pyexadis_paths
 if os.path.abspath(path) not in sys.path]

np.set_printoptions(threshold=20, edgeitems=5)

try:
    import pyexadis
    from pyexadis_base import (
        ExaDisNet, DisNetManager, SimulateNetwork,  # If want multiple values in csv always use SimulateNetwork

        CalForce, MobilityLaw, TimeIntegration,
        Collision, Topology, Remesh
    )
except ImportError:
    raise ImportError("Cannot import pyexadis")



# Helpers (small utility functions used throughout the simulation)

def voigt_sym_to_tensor(v6):
    # Voigt order: [xx, yy, zz, yz, xz, xy]

    v6 = np.asarray(v6, dtype=float).ravel()
    return np.array([
        [v6[0], v6[5], v6[4]],
        [v6[5], v6[1], v6[3]],
        [v6[4], v6[3], v6[2]]
    ], dtype=float)


def von_mises_stress_from_voigt(v6):
    S = voigt_sym_to_tensor(v6)
    Sdev = S - np.trace(S) / 3.0 * np.eye(3)
    return np.sqrt(1.5 * np.tensordot(Sdev, Sdev, axes=2))


def von_mises_strain_from_voigt(v6):
    E = voigt_sym_to_tensor(v6)
    Edev = E - np.trace(E) / 3.0 * np.eye(3)
    return np.sqrt((2.0 / 3.0) * np.tensordot(Edev, Edev, axes=2))


class StopSimulation(RuntimeError):
    """Internal exception to stop the simulation cleanly."""
    pass



# Cyclic strain-rate driver

class CyclicStrainDriver(SimulateNetwork):

    # One cycle:
    #     0 -> +eps_max -> -eps_max -> 0

    # Strain control is imposed in one direction only (y in our case)
    # Stress update follows isotropic linear elasticity:
    #     dEe = dE - dEp
    #     dstress = lambda * tr(dEe) * I + 2 * mu * dEe


    def __init__(self, *args,
                 eps_max,
                 strain_rate,
                 num_cycles,
                 load_axis='y',
                 state=None,
                 **kwargs):

        if state is None:
            raise ValueError("state dictionary must be provided")

        # Separate frequency for custom CSV/DAT output
        self.custom_write_freq = kwargs.pop("custom_write_freq", 1)

        # Passing state explicitly to parent constructor
        super().__init__(*args, state=state, **kwargs)

        self.MU = float(state["mu"])
        self.NU = float(state["nu"])
        self.LA = 2.0 * self.MU * self.NU / (1.0 - 2.0 * self.NU)

        self.eps_max = float(eps_max)          # e.g. 0.0035 for 0.35%
        self.strain_rate = float(strain_rate)  # 1/s
        self.num_cycles = int(num_cycles)

        axis_map = {'x': 0, 'y': 1, 'z': 2}
        if load_axis not in axis_map:
            raise ValueError("load_axis must be 'x', 'y', or 'z'")
        self.load_axis = load_axis
        self.comp = axis_map[load_axis]

        # Time bookkeeping
        self._time_cum = 0.0
        self._ep_eq_cum = 0.0
        self._step_fallback = 0

        # One cycle:
        # 0 -> +eps_max
        # +eps_max -> -eps_max
        # -eps_max -> 0
        self.t_leg = self.eps_max / self.strain_rate
        self.t_cycle = 4.0 * self.t_leg
        self.t_total = self.num_cycles * self.t_cycle

        # Output
        self._out_dir = self.write_dir
        self._csv_main = os.path.join(self._out_dir, "cyclic_strain_control.csv")
        self._dat_main = os.path.join(self._out_dir, "cyclic_strain_control.dat")

        self._wrote_header_csv = os.path.exists(self._csv_main)
        self._wrote_header_dat = os.path.exists(self._dat_main)

    def strain_target_scalar(self, t):

        # Signed imposed axial strain history for one cycle:
        #     0 -> +eps_max -> -eps_max -> 0
        # repeated for num_cycles.

        if t <= 0.0:
            return 0.0
        if t >= self.t_total:
            return 0.0

        tau = t % self.t_cycle

        # Segment 1: 0 -> +eps_max
        if tau < self.t_leg:
            return self.strain_rate * tau

        # Segment 2: +eps_max -> -eps_max
        elif tau < 3.0 * self.t_leg:
            return self.eps_max - self.strain_rate * (tau - self.t_leg)

        # Segment 3: -eps_max -> 0
        else:
            return -self.eps_max + self.strain_rate * (tau - 3.0 * self.t_leg)

    def build_strain_increment_voigt(self, deps_scalar):

        # Uniaxial imposed strain increment in chosen direction.
        # For y-loading: [0, deps, 0, 0, 0, 0]

        dE = np.zeros(6, dtype=float)
        dE[self.comp] = deps_scalar
        return dE

    def step_update_response(self, N: DisNetManager, state: dict):

        # built-in loading mode string as per Exadis
        if self.loading_mode != 'stress':
            return super().step_update_response(N, state)

        # Getting plastic strain increment, plastic spin increment, density
        dEp_T, dWp_vec, density = N.get_disnet(ExaDisNet).net.get_plastic_strain()
        state["density"] = float(density)

        # Converting ExaDiS plastic strain tensor increment to Voigt:
        # [xx, yy, zz, yz, xz, xy]
        dEp = np.array(dEp_T, dtype=float).ravel()[[0, 4, 8, 5, 2, 1]]

        # Plastic spin components stored as [yz, xz, xy]
        dWp = np.array(dWp_vec, dtype=float).ravel()[[5, 2, 1]]

        state["dEp"] = dEp
        state["dWp"] = dWp

        dt_raw = float(state["dt"])

        # Clamping final step so we land exactly at t_total
        rem = self.t_total - self._time_cum
        dt_eff = max(0.0, min(dt_raw, rem))
        state["dt_eff"] = dt_eff

        # Imposed axial strain target at old and new times
        eps_old = self.strain_target_scalar(self._time_cum)
        eps_new = self.strain_target_scalar(self._time_cum + dt_eff)
        deps_scalar = eps_new - eps_old

        # Advance user time
        self._time_cum += dt_eff
        state["time_user"] = self._time_cum
        state["eps_target_scalar"] = eps_new

        # Imposed total strain increment
        dE = self.build_strain_increment_voigt(deps_scalar)

        # Elastic strain increment
        dEe = dE - dEp

        # Stress increment:
        # dstress = lambda * tr(dEe) * [1,1,1,0,0,0] + 2 mu dEe
        dstress = (
            self.LA * np.sum(dEe[0:3]) * np.array([1, 1, 1, 0, 0, 0], dtype=float)
            + 2.0 * self.MU * dEe
        )

        state["dEe"] = dEe.copy()
        state["applied_stress"] = state.get("applied_stress", np.zeros(6, dtype=float)) + dstress
        state["Etot"] = state.get("Etot", np.zeros(6, dtype=float)) + dE

        # Scalars
        state["stress"] = float(von_mises_stress_from_voigt(state["applied_stress"]))
        state["strain"] = float(von_mises_strain_from_voigt(state["Etot"]))

        return state

    def step_end(self, N: DisNetManager, state: dict):
        super().step_end(N, state)
        if self._time_cum >= self.t_total - 1.0e-30:
            raise StopSimulation(
                f"Completed cyclic strain loading: time={self._time_cum:.6e} s, "
                f"cycles={self.num_cycles}"
            )

    def step_write_files(self, N: DisNetManager, state: dict):

        # Custom CSV/DAT writing frequency is controlled by self.custom_write_freq,

        step = int(state.get("istep", state.get("step", self._step_fallback + 1)))
        if self.custom_write_freq and (step % int(self.custom_write_freq) != 0):
            return

        os.makedirs(self._out_dir, exist_ok=True)

        Etot = np.asarray(state.get("Etot", np.zeros(6)), dtype=float).ravel()
        dEp  = np.asarray(state.get("dEp",  np.zeros(6)), dtype=float).ravel()
        s6   = np.asarray(state.get("applied_stress", np.zeros(6)), dtype=float).ravel()

        dt_eff = float(state.get("dt_eff", state.get("dt", 0.0)))
        time_user = float(state.get("time_user", self._time_cum))
        cycle_index = int(time_user / self.t_cycle)
        cycle_time = time_user - cycle_index * self.t_cycle

        strain_eq = float(state.get("strain", 0.0))
        sigma_vm = float(state.get("stress", 0.0))
        eps_target = float(state.get("eps_target_scalar", 0.0))
        density = float(state.get("density", 0.0))

        # Plastic spin tensor from [yz, xz, xy]
        wyz, wxz, wxy = state["dWp"]
        dWp = np.array([
            [0.0,   wxy,  wxz],
            [-wxy,  0.0,  wyz],
            [-wxz, -wyz,  0.0]
        ], dtype=float)

        dEp_tensor = voigt_sym_to_tensor(dEp)
        Dp = dEp_tensor / max(dt_eff, 1.0e-30)
        Lp = (dEp_tensor + dWp) / max(dt_eff, 1.0e-30)

        epdot_eq = float(np.sqrt((2.0 / 3.0) * np.tensordot(Dp, Dp, axes=2)))
        self._ep_eq_cum += epdot_eq * dt_eff

        if not self._wrote_header_csv:
            with open(self._csv_main, "w", newline="") as f:
                w = csv.writer(f)
                w.writerow([
                    "step", "time_user(s)", "dt_eff(s)", "cycle_index", "cycle_time(s)",
                    "strain_eq", "sigma_vm(Pa)", "eps_target(-)",
                    "s_xx(Pa)", "s_yy(Pa)", "s_zz(Pa)", "s_yz(Pa)", "s_xz(Pa)", "s_xy(Pa)",
                    "Etot_yy",
                    "density(1/m^2)",
                    "Lpxx", "Lpxy", "Lpxz", "Lpyx", "Lpyy", "Lpyz", "Lpzx", "Lpzy", "Lpzz",
                    "epdot_eq(1/s)", "ep_eq(-)"
                ])
            self._wrote_header_csv = True

        if not self._wrote_header_dat:
            with open(self._dat_main, "w") as f:
                f.write(
                    "# step time_user(s) dt_eff(s) cycle_index cycle_time(s) "
                    "strain_eq sigma_vm(Pa) eps_target(-) "
                    "s_xx(Pa) s_yy(Pa) s_zz(Pa) s_yz(Pa) s_xz(Pa) s_xy(Pa) "
                    "Etot_yy "
                    "density(1/m^2) "
                    "Lpxx Lpxy Lpxz Lpyx Lpyy Lpyz Lpzx Lpzy Lpzz "
                    "epdot_eq(1/s) ep_eq(-)\n"
                )
            self._wrote_header_dat = True

        row = [
            step, time_user, dt_eff, cycle_index, cycle_time,
            strain_eq, sigma_vm, eps_target,
            s6[0], s6[1], s6[2], s6[3], s6[4], s6[5],
            Etot[1],      # Etot_yy
            density,
            *list(Lp.ravel(order='C')),
            epdot_eq, self._ep_eq_cum
        ]

        with open(self._csv_main, "a", newline="") as f:
            csv.writer(f).writerow(row)

        with open(self._dat_main, "a") as f:
            f.write(" ".join([
                f"{step:d}",
                f"{time_user:.8e}",
                f"{dt_eff:.8e}",
                f"{cycle_index:d}",
                f"{cycle_time:.8e}",
                f"{strain_eq:.8e}",
                f"{sigma_vm:.8e}",
                f"{eps_target:.8e}",
                f"{s6[0]:.8e}",
                f"{s6[1]:.8e}",
                f"{s6[2]:.8e}",
                f"{s6[3]:.8e}",
                f"{s6[4]:.8e}",
                f"{s6[5]:.8e}",
                f"{Etot[1]:.8e}",
                f"{density:.8e}",
                *[f"{x:.6e}" for x in Lp.ravel(order='C')],
                f"{epdot_eq:.6e}",
                f"{self._ep_eq_cum:.6e}",
            ]) + "\n")

# Main

def main():
    pyexadis.initialize()

    try:

        # Global state

        state = {
            "crystal": 'fcc',
            "burgmag": 2.55e-10,
            "mu": 54.6e9,
            "nu": 0.324,
            "a": 6.0,
            "maxseg": 4500.0,
            "minseg": 900.0,
            "rtol": 11.0,
            "rann": 8.0,
            "nextdt": 4.0e-11,
            "maxdt": 2.7e-10,
            "applied_stress": np.zeros(6, dtype=float),
            "Etot": np.zeros(6, dtype=float),
        }


        # Initial microstructure: 15 um box

        Lbox = 58824.0
        G = ExaDisNet().generate_line_config(
            state["crystal"], Lbox,
            num_lines=15 * 12,  # Change this for dislocation density
            theta=[0.0, 60.0, 90.0],
            maxseg=state["maxseg"],
            seed=1234
        )
        N = DisNetManager(G)

        vis = None       # keep this on if want visualization


        # Common modules

        calforce = CalForce(
            force_mode='SUBCYCLING_MODEL',     # Calculates force on dislocations
            state=state,
            Ngrid=64,
            cell=N.cell
        )
        mobility = MobilityLaw(
            mobility_law='FCC_0',              # Calculates velocity
            state=state,
            Medge=64103.0,
            Mscrew=64103.0,
            vmax=2500.0
        )
        timeint = TimeIntegration(
            integrator='Subcycling',
            rgroups=[0.0, 100.0, 600.0, 1500.0],          # Updates the position at every step
            state=state,
            force=calforce,
            mobility=mobility
        )
        collision = Collision(collision_mode='Retroactive', state=state)      # Handles collision and interaction effects
        topology = Topology(
            topology_mode='TopologyParallel',
            state=state,
            force=calforce,
            mobility=mobility
        )
        remesh = Remesh(remesh_rule='LengthBased', state=state)
        cross_slip = None                   # No cross slip in our case so none


        # Cyclic strain-rate control (NO RELAXATION)

        eps_max = 0.0035        # 0.35% , change this line only for target strain
        strain_rate = 5.0e3    # 1/s , change this line only for rate
        num_cycles = 1
        load_axis = 'y'

        print("Starting cyclic strain-controlled loading WITHOUT prior relaxation...")

        sim_cyclic = CyclicStrainDriver(
            calforce=calforce,
            mobility=mobility,
            timeint=timeint,
            collision=collision,
            topology=topology,
            remesh=remesh,
            cross_slip=cross_slip,
            vis=vis,

            # Built-in driver mode kept valid so ExaDiS control initializes
            loading_mode='stress',
            applied_stress=np.zeros(6, dtype=float),

            max_step=200000,
            burgmag=state["burgmag"],
            state=state,

            print_freq=200,
            plot_freq=0,

            # Built-in config/restart writing frequency
            write_freq=1000,

            write_dir='cyclicstrain_nopre_relaxation_fastreduced22', # change this everytime for new simulation

            # Custom CSV/DAT every step
            custom_write_freq=10,  # Decides print frequency in csv file

            eps_max=eps_max,
            strain_rate=strain_rate,
            num_cycles=num_cycles,
            load_axis=load_axis
        )

        sim_cyclic.run(N, state)

    except StopSimulation as e:
        print(str(e))

    finally:
        pyexadis.finalize()


if __name__ == "__main__":
    main()

In [ ]:
# ANN CODE FOR STRESS CONTROLLED LOADING

# Here , We train two separate neural networks on this data:
# 1. DpNet  : predicts the PLASTIC STRAIN RATE tensor (Dp) given current
#                  stress state, dislocation density, and loading info
# 2. RhoNet : predicts the DISLOCATION DENSITY EVOLUTION RATE given the
#                  same inputs plus the current time step size

import os
import re
import json
import time
import zipfile
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, Dataset # Data loading utilities


# GLOBAL CONFIG
# DDD database. Built from data/raw/ on first use so the notebook runs
# from a fresh clone of the repository.
from pathlib import Path as _Path
import zipfile as _zipfile

REPO_ROOT = _Path.cwd()
if not (REPO_ROOT / "data" / "raw").is_dir():
    REPO_ROOT = REPO_ROOT.parent

RAW_DIR  = REPO_ROOT / "data" / "raw"
ZIP_PATH = str(REPO_ROOT / "data" / "SimulationCsvs.zip")

if not _Path(ZIP_PATH).exists():
    _csvs = sorted(RAW_DIR.glob("*.csv"))
    if not _csvs:
        raise FileNotFoundError(f"No CSV files in {RAW_DIR}")
    with _zipfile.ZipFile(ZIP_PATH, "w", _zipfile.ZIP_DEFLATED) as _zf:
        for _c in _csvs:
            _zf.write(_c, arcname=_c.name)
    print(f"Built {ZIP_PATH} from {len(_csvs)} runs")

# Folders where trained model outputs will be saved
DATASET_DIR = "data/processed/ann_v7"
NN_OUT_DIR  = "results/ann_v7"
os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(NN_OUT_DIR, exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)               # ensures results are reproducible
torch.manual_seed(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

N_RESAMPLE = 1000  # Resampled each simulation to 1000 evenly spaced time points

N_VAL_RUNS  = 8
N_TEST_RUNS = 8        # Number of validation and testing runs

# Normalization Parameters
STRESS_SCALE   = 55.0e6  # Dividing stress by 55 MPa which is the maximum value
LOGRHO_SHIFT   = 12.0    # Subtract 12 from log10(density) to centers around 0
SDOT_LOG_SHIFT = 13.76

## NEURAL NETWORK ARCHITECTURE (Dp branch)
DP_HIDDEN_DIMS   = [128, 256, 128, 64]
DP_ACTIVATION    = "SiLU"
DP_DROPOUT       = 0.05
DP_USE_BATCHNORM = False

# Rho model config
# rho input now has one extra feature: dt
RHO_HIDDEN_DIMS   = [128, 128, 64, 32]
RHO_ACTIVATION    = "SiLU"
RHO_DROPOUT       = 0.05
RHO_USE_BATCHNORM = False

# Training Parameters
BATCH_SIZE       = 512
ROLL_BATCH_SIZE  = 128
MAX_EPOCHS       = 2500
LR_INIT          = 3e-4
T0               = 100
T_MULT           = 2
LR_MIN_COSINE    = 1e-6  # Minimum learning rate at the bottom of each cycle
ES_PATIENCE      = 250
WEIGHT_DECAY     = 1e-4
GRAD_CLIP        = 5.0    # Clipping gradient norm to prevent exploding gradients

# noise
INPUT_NOISE_STD  = 0.01    # improves generalization
HUBER_DELTA      = 1.0     # Huber loss delta behaving like MSE for small errors, MAE for large errors

# rollout-training
WINDOW_LEN          = 24   # Number of time steps in each rollout window
WINDOW_STRIDE       = 12   # Step between window start positions
ROLL_START_EPOCH    = 1

# Dp weights
W_STEP_DP           = 1.0
W_ROLL_DP           = 0.35

# Rho weights
W_STEP_RHO          = 1.0
W_ROLL_RHO          = 0.50

MAX_WINDOWS_PER_RUN = None

# COLUMN NAME DEFINITIONS
LP_COLS  = ["Lpxx","Lpxy","Lpxz","Lpyx","Lpyy","Lpyz","Lpzx","Lpzy","Lpzz"]
SIG_COLS = ["s_xx(Pa)","s_yy(Pa)","s_zz(Pa)","s_yz(Pa)","s_zx(Pa)","s_xy(Pa)"]
EP_COL   = "ep_eq(-)"
RHO_COL  = "density(1/m^2)"

INPUT_LABELS_DP = ['σ_xx','σ_yy','σ_zz','σ_yz','σ_zx','σ_xy',
                   'σ_dot','log10ρ','loading','εp_cum']

INPUT_LABELS_RHO = ['σ_xx','σ_yy','σ_zz','σ_yz','σ_zx','σ_xy',
                    'σ_dot','log10ρ','loading','εp_cum','dt']
#Output labels for the plastic strain rate tensor components
DP_OUTPUT_LABELS_5 = ['Dp_xx','Dp_yy','Dp_yz','Dp_xz','Dp_xy']
DP_OUTPUT_LABELS_6 = ['Dp_xx','Dp_yy','Dp_zz','Dp_yz','Dp_xz','Dp_xy']



def set_all_seeds(seed: int):      # Sets random seeds for numpy, PyTorch CPU, and PyTorch GPU for reproducibility.
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def dp6_to_epdot_eq_np(Dp6):     # Computes the equivalent (scalar) plastic strain rate from Dp
    T = np.array([
        [Dp6[0], Dp6[5], Dp6[4]],
        [Dp6[5], Dp6[1], Dp6[3]],
        [Dp6[4], Dp6[3], Dp6[2]],
    ], dtype=np.float64)
    return np.sqrt((2.0 / 3.0) * np.sum(T * T))


def dp5_to_epdot_eq_torch(dp5_phys):
    dxx = dp5_phys[..., 0]
    dyy = dp5_phys[..., 1]
    dzz = -(dxx + dyy)
    dyz = dp5_phys[..., 2]     # same as above but with pytorch tensors
    dxz = dp5_phys[..., 3]
    dxy = dp5_phys[..., 4]

    TT = dxx*dxx + dyy*dyy + dzz*dzz + 2.0*(dxy*dxy + dxz*dxz + dyz*dyz)
    epdot = torch.sqrt((2.0 / 3.0) * torch.clamp(TT, min=1e-40)) # clamping prevents negative values under sqrt due to floating point errors
    return epdot


def reconstruct_Y6_from_Y5(Y5_phys):
    Dp_xx = Y5_phys[:, 0:1]
    Dp_yy = Y5_phys[:, 1:2]
    Dp_zz = -(Dp_xx + Dp_yy)       # Enforcing plastic incompressibility
    Dp_yz = Y5_phys[:, 2:3]
    Dp_xz = Y5_phys[:, 3:4]
    Dp_xy = Y5_phys[:, 4:5]
    return np.hstack([Dp_xx, Dp_yy, Dp_zz, Dp_yz, Dp_xz, Dp_xy])



# DATASET BUILDING

def parse_filename(name: str):  # Extracting simulation metadata from the CSV filename.

    stem = Path(name).stem
    loading = 0 if re.search(r'(?i)uniaxial', stem) else \  # Detecting loading type from filename keywords
              1 if re.search(r'(?i)shear', stem) else -1

    nums = [float(x) for x in re.findall(  # Extracting all numbers from the filename
        r'[\d]+(?:\.[\d]+)?(?:e[+-]?[\d]+)?', stem, re.IGNORECASE
    )]
# Classifying numbers by their magnitude into the correct physical quantity
    target_mpa = sigma_dot = rho0 = None
    for n in nums:
        if 10 <= n <= 200 and target_mpa is None:  # Target stress in Pa
            target_mpa = n * 1e6
        elif 1e12 <= n <= 1e14 and sigma_dot is None: # stress rate in Pa/s
            sigma_dot = n
        elif 1e10 <= n <= 1e13 and rho0 is None:
            rho0 = n

    return dict(
        stem=stem,
        loading=loading,
        target_Pa=target_mpa,
        sigma_dot=sigma_dot,
        rho0=rho0
    )


def resample_run(df, meta, n_points):  # Resamples one DDD simulation run to a fixed number of evenly-spaced time points.

    t = df["time(s)"].values.astype(float)
    required = SIG_COLS + LP_COLS + [EP_COL, RHO_COL]
# Validation checks
    if len(np.unique(t)) < 2:      # Need at least 2 distinct time points
        return None
    if not all(c in df.columns for c in required):  # If any required column is missing
        return None
    if meta["sigma_dot"] is None or meta["rho0"] is None:
        return None
# Creating uniform time grid
    t_new = np.linspace(t[0], t[-1], n_points)
    dt_uniform = t_new[1] - t_new[0] if len(t_new) > 1 else 0.0

    def interp(cols):  # Linearly interpolates multiple columns onto the new time grid.
        arr = df[cols].values.astype(float)
        out = np.zeros((n_points, len(cols)), float)
        for j in range(arr.shape[1]):
            f = interp1d(
                t, arr[:, j],
                kind="linear",
                bounds_error=False,
                fill_value=(arr[0, j], arr[-1, j]),
            )
            out[:, j] = f(t_new)
        return out

    sig    = interp(SIG_COLS)
    ep_cum = interp([EP_COL])      # Interpolating stress and accumulated plastic strain

    rho = interp([RHO_COL]).reshape(-1)
    rho = np.maximum(rho, 1e6)
    log10rho = np.log10(rho)          # Interpolating density and converting to log10 scale

    # Computing increment of log10(rho) between consecutive time steps which is what Rhonet will learn
    dlog10rho = np.zeros(n_points, dtype=float)
    dlog10rho[:-1] = log10rho[1:] - log10rho[:-1]
    dlog10rho[-1]  = dlog10rho[-2] if n_points > 1 else 0.0

    # onverting increment to rate
    if dt_uniform > 0:
        dlog10rho_rate = dlog10rho / dt_uniform
    else:
        dlog10rho_rate = np.zeros_like(dlog10rho)

    Lp_raw = interp(LP_COLS)
    Lp_mat = Lp_raw.reshape(-1, 3, 3)

    Dp_mat = 0.5 * (Lp_mat + Lp_mat.transpose(0, 2, 1))  # Dymmetric part of Lp
    Dp = np.stack([
        Dp_mat[:, 0, 0],
        Dp_mat[:, 1, 1],
        Dp_mat[:, 2, 2],     # converting Dp from 3x3 to 6-component Voigt notation
        Dp_mat[:, 1, 2],
        Dp_mat[:, 0, 2],
        Dp_mat[:, 0, 1],
    ], axis=1)
    # Here we are Returning all processed quantities as a dictionary
    return dict(
        sig            = sig,
        ep_cum         = ep_cum,
        rho            = rho,
        log10rho       = log10rho,
        dlog10rho      = dlog10rho,
        dlog10rho_rate = dlog10rho_rate,
        Dp             = Dp,
        sigma_dot      = np.full(n_points, meta["sigma_dot"], float),
        rho0           = np.full(n_points, meta["rho0"], float),
        loading        = np.full(n_points, float(meta["loading"]), float),
        time           = t_new,
        dt             = np.full(n_points, dt_uniform, float),
        stem           = meta["stem"],
        n_orig         = len(df),
        strat_key      = f"load{meta['loading']}_rho{meta['rho0']:.2e}",
    )

# Splits all simulation runs into train/val/test sets in a stratified way.
# val and test sets to cover all types of simulations (loading types and density levels), not just the easiest ones

def exact_grouped_stratified_split(run_data, n_val=8, n_test=8, seed=42):
    rng = np.random.default_rng(seed)
    # Run indices have been grouped by their stratification key
    strata = defaultdict(list)
    for i, rd in enumerate(run_data):
        strata[rd["strat_key"]].append(i)

    print(f"\nStrata found ({len(strata)}):")
    for k, v in sorted(strata.items()):
        print(f"  {k}: {len(v)} runs")
    # Shuffling within each stratum for randomness
    shuffled = {}
    for k, idxs in strata.items():
        arr = np.array(idxs)
        rng.shuffle(arr)
        shuffled[k] = arr.tolist()

    keys = sorted(strata.keys())

    val_counts = {k: 1 for k in keys}
    test_counts = {k: 1 for k in keys}

    # Remaining slots to fill
    extra_val = n_val - len(keys)
    extra_test = n_test - len(keys)

    if extra_val < 0 or extra_test < 0:
        raise ValueError("n_val and n_test must be at least number of strata.")

    def assign_extras(counts_a, counts_b, n_extra):
        if n_extra <= 0:
            return
        for _ in range(n_extra):
            best_k = None
            best_left = -1
            for k in keys:
                total = len(shuffled[k])
                used = counts_a[k] + counts_b[k]
                left = total - used
                if left > best_left:
                    best_left = left
                    best_k = k
            if best_k is None or best_left <= 0:
                raise RuntimeError("Not enough runs to allocate exact split.")
            counts_a[best_k] += 1

    assign_extras(val_counts, test_counts, extra_val)
    assign_extras(test_counts, val_counts, extra_test)

    idx_train, idx_val, idx_test = [], [], []

    for k in keys:
        arr = shuffled[k]
        nv = val_counts[k]
        nt = test_counts[k]
        n = len(arr)

        if nv + nt >= n:
            raise RuntimeError(f"Stratum {k} too small for requested exact split.")

        idx_val.extend(arr[:nv])
        idx_test.extend(arr[nv:nv+nt])
        idx_train.extend(arr[nv+nt:])

    return idx_train, idx_val, idx_test, val_counts, test_counts


def assemble(run_data, idxs): # Assembles individual run data dictionaries into large numpy arrays
    # Dp branch
    sigs_dp, sdots_dp, logrhos_dp, loads_dp, eps_dp = [], [], [], [], []
    Y_dp = []

    # Rho branch
    sigs_rho, sdots_rho, logrhos_rho, loads_rho, eps_rho, dt_rho = [], [], [], [], [], []
    Y_rho = []

    for i in idxs:
        rd = run_data[i]

        # Dp branch inputs
        sigs_dp.append(rd["sig"])
        sdots_dp.append(rd["sigma_dot"][:, None])
        logrhos_dp.append(rd["log10rho"][:, None])
        loads_dp.append(rd["loading"][:, None])
        eps_dp.append(rd["ep_cum"])

        Y_dp.append(np.column_stack([
            rd["Dp"][:, 0],
            rd["Dp"][:, 1],      # Dp outputs
            rd["Dp"][:, 3],
            rd["Dp"][:, 4],
            rd["Dp"][:, 5],
        ]))

        # Rho branch inputs
        sigs_rho.append(rd["sig"])
        sdots_rho.append(rd["sigma_dot"][:, None])
        logrhos_rho.append(rd["log10rho"][:, None])
        loads_rho.append(rd["loading"][:, None])
        eps_rho.append(rd["ep_cum"])
        dt_rho.append(rd["dt"][:, None])


        Y_rho.append(rd["dlog10rho_rate"][:, None])

    X_dp = np.hstack([
        np.vstack(sigs_dp),
        np.vstack(sdots_dp),       # Stacking all runs into single arrays
        np.vstack(logrhos_dp),
        np.vstack(loads_dp),
        np.vstack(eps_dp),
    ])

    X_rho = np.hstack([
        np.vstack(sigs_rho),
        np.vstack(sdots_rho),
        np.vstack(logrhos_rho),
        np.vstack(loads_rho),
        np.vstack(eps_rho),
        np.vstack(dt_rho),
    ])

    Y_dp  = np.vstack(Y_dp)
    Y_rho = np.vstack(Y_rho)

    return X_dp.astype(np.float32), Y_dp.astype(np.float32), X_rho.astype(np.float32), Y_rho.astype(np.float32)


def save_run_sequences(run_data, idx_train, idx_val, idx_test, out_dir): # Saves all simulation data in a flat CSV format where each row is one
    time point from one run
    split_map = {}
    for i in idx_train:
        split_map[run_data[i]["stem"]] = "train"
    for i in idx_val:
        split_map[run_data[i]["stem"]] = "val"
    for i in idx_test:
        split_map[run_data[i]["stem"]] = "test"

    rows = []
    for rd in run_data:
        stem = rd["stem"]
        split = split_map[stem]
        n = rd["sig"].shape[0]

        for k in range(n):
            rows.append({
                "stem": stem,
                "split": split,
                "k": k,
                "time": rd["time"][k],
                "dt": rd["dt"][k],
                "s_xx": rd["sig"][k, 0],
                "s_yy": rd["sig"][k, 1],
                "s_zz": rd["sig"][k, 2],
                "s_yz": rd["sig"][k, 3],
                "s_zx": rd["sig"][k, 4],
                "s_xy": rd["sig"][k, 5],
                "sigma_dot": rd["sigma_dot"][k],
                "rho0": rd["rho0"][k],
                "rho_true": rd["rho"][k],
                "log10rho_true": rd["log10rho"][k],
                "dlog10rho_true": rd["dlog10rho"][k],
                "dlog10rho_rate_true": rd["dlog10rho_rate"][k],
                "loading": rd["loading"][k],
                "ep_eq_cum": rd["ep_cum"][k, 0],
                "Dp_xx": rd["Dp"][k, 0],
                "Dp_yy": rd["Dp"][k, 1],
                "Dp_zz": rd["Dp"][k, 2],
                "Dp_yz": rd["Dp"][k, 3],
                "Dp_xz": rd["Dp"][k, 4],
                "Dp_xy": rd["Dp"][k, 5],
            })

    df_seq = pd.DataFrame(rows)
    seq_path = os.path.join(out_dir, "run_sequences_v7_rho_improved.csv")
    df_seq.to_csv(seq_path, index=False)
    print(f"Saved run sequences → {seq_path}")
    return seq_path


def build_dataset_v7():

  """
    Main dataset building pipeline:
    1. Reads all CSV files from the zip archive
    2. Parses metadata from each filename
    3. Resamples each run to N_RESAMPLE uniform time points
    4. Splits runs into train/val/test sets (stratified)
    5. Assembles numpy arrays and saves them
    6. Computes and saves normalization values
    """
    print(f"Reading {ZIP_PATH} ...")
    run_data = []

    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        # Get all CSV filenames in the zip
        csv_names = sorted([
            n for n in zf.namelist()
            if n.lower().endswith(".csv") and not n.startswith("__")
        ])
        print(f"Found {len(csv_names)} CSV files")

        for name in csv_names:
            meta = parse_filename(name)
            try:
                with zf.open(name) as f:
                    df = pd.read_csv(f, comment="#")
                df.columns = [c.strip() for c in df.columns]

                rd = resample_run(df, meta, N_RESAMPLE)
                if rd is not None:
                    run_data.append(rd)
                    print(f"  ✓ {meta['stem'][:52]:52s} orig={rd['n_orig']:6d} strat={rd['strat_key']}")
            except Exception as e:
                print(f"  ✗ {meta['stem']}: {e}")

    print(f"\nTotal runs: {len(run_data)}")

    # Spliting into train/val/test using stratified splitting
    idx_train, idx_val, idx_test, val_counts, test_counts = exact_grouped_stratified_split(
        run_data, n_val=N_VAL_RUNS, n_test=N_TEST_RUNS, seed=RANDOM_SEED
    )

    print(f"\nExact split: train={len(idx_train)}  val={len(idx_val)}  test={len(idx_test)} runs")

    print("\nVal counts per stratum:")
    for k in sorted(val_counts):
        print(f"  {k}: {val_counts[k]}")

    print("\nTest counts per stratum:")
    for k in sorted(test_counts):
        print(f"  {k}: {test_counts[k]}")

    def strat_set(idxs):
        return set(run_data[i]["strat_key"] for i in idxs)

    all_strata = strat_set(range(len(run_data)))
    tr_s, va_s, te_s = strat_set(idx_train), strat_set(idx_val), strat_set(idx_test)
    print(f"\nStrata coverage — train:{len(tr_s)}/{len(all_strata)}  val:{len(va_s)}/{len(all_strata)}  test:{len(te_s)}/{len(all_strata)}")

    #  Assembling large numpy arrays from selected runs
    X_train_dp_raw, Y_train_dp_raw, X_train_rho_raw, Y_train_rho_raw = assemble(run_data, idx_train)
    X_val_dp_raw,   Y_val_dp_raw,   X_val_rho_raw,   Y_val_rho_raw   = assemble(run_data, idx_val)
    X_test_dp_raw,  Y_test_dp_raw,  X_test_rho_raw,  Y_test_rho_raw  = assemble(run_data, idx_test)

    print(f"\nShapes:")
    print(f"  X_train_dp ={X_train_dp_raw.shape}  Y_train_dp ={Y_train_dp_raw.shape}")
    print(f"  X_val_dp   ={X_val_dp_raw.shape}    Y_val_dp   ={Y_val_dp_raw.shape}")
    print(f"  X_test_dp  ={X_test_dp_raw.shape}   Y_test_dp  ={Y_test_dp_raw.shape}")
    print(f"  X_train_rho={X_train_rho_raw.shape}  Y_train_rho={Y_train_rho_raw.shape}")
    print(f"  X_val_rho  ={X_val_rho_raw.shape}    Y_val_rho  ={Y_val_rho_raw.shape}")
    print(f"  X_test_rho ={X_test_rho_raw.shape}   Y_test_rho ={Y_test_rho_raw.shape}")

    #Computing output normalization statistics from training data only
    Y_dp_std = Y_train_dp_raw.std(axis=0).astype(np.float64)
    Y_dp_std[Y_dp_std < 1e-40] = 1.0  # Prevent division by zero


    Y_rho_std = float(Y_train_rho_raw.std())
    if Y_rho_std < 1e-40:
        Y_rho_std = 1.0

    EP_MAX = float(X_train_dp_raw[:, 9].max())

    # dt normalization stats for rho branch
    dt_log_train = np.log10(np.maximum(X_train_rho_raw[:, 10], 1e-40))
    DT_LOG_MEAN = float(dt_log_train.mean())
    DT_LOG_STD  = float(dt_log_train.std())
    if DT_LOG_STD < 1e-12:
        DT_LOG_STD = 1.0
    # we bundle all normalization stats into a dictionary for saving
    stats = {
        "Y_dp_std": Y_dp_std.tolist(),
        "Y_rho_std": Y_rho_std,
        "EP_MAX": EP_MAX,
        "DT_LOG_MEAN": DT_LOG_MEAN,
        "DT_LOG_STD": DT_LOG_STD,
        "n_inputs_dp": 10,
        "n_inputs_rho": 11,
        "N_RESAMPLE": N_RESAMPLE,
        "input_order_dp": ["s_xx","s_yy","s_zz","s_yz","s_zx","s_xy",
                           "sigma_dot","log10rho","loading","ep_eq_cum"],
        "input_order_rho": ["s_xx","s_yy","s_zz","s_yz","s_zx","s_xy",
                            "sigma_dot","log10rho","loading","ep_eq_cum","dt"],
        "dp_output_order": ["Dp_xx","Dp_yy","Dp_yz","Dp_xz","Dp_xy"],
        "rho_output_order": ["dlog10rho_rate"],
        "window_len": WINDOW_LEN,
        "window_stride": WINDOW_STRIDE,
        "split_info": {
            "train_runs": len(idx_train),
            "val_runs": len(idx_val),
            "test_runs": len(idx_test),
        },
    }

     # File paths for saving
    npz_path   = os.path.join(DATASET_DIR, "ddd_dataset_v7_rho_improved.npz")
    stats_path = os.path.join(DATASET_DIR, "norm_stats_v7_rho_improved.json")
    split_path = os.path.join(DATASET_DIR, "run_split_v7_rho_improved.csv")

    np.savez_compressed(
        npz_path,
        X_train_dp_raw=X_train_dp_raw,
        X_val_dp_raw=X_val_dp_raw,            # All arrays have been saved in one compressed file
        X_test_dp_raw=X_test_dp_raw,
        Y_train_dp_raw=Y_train_dp_raw,
        Y_val_dp_raw=Y_val_dp_raw,
        Y_test_dp_raw=Y_test_dp_raw,
        X_train_rho_raw=X_train_rho_raw,
        X_val_rho_raw=X_val_rho_raw,
        X_test_rho_raw=X_test_rho_raw,
        Y_train_rho_raw=Y_train_rho_raw,
        Y_val_rho_raw=Y_val_rho_raw,
        Y_test_rho_raw=Y_test_rho_raw,
    )

    with open(stats_path, "w") as f:
        json.dump(stats, f, indent=2)

    # Saving split assignment (which run belongs to which split) as CSV
    rows = []
    for label, idxs in [("train", idx_train), ("val", idx_val), ("test", idx_test)]:
        for i in idxs:
            rows.append({
                "stem": run_data[i]["stem"],
                "split": label,
                "strat_key": run_data[i]["strat_key"]
            })
    pd.DataFrame(rows).to_csv(split_path, index=False)

    # Save flat time-series CSV used for rollout window datasets
    seq_path = save_run_sequences(run_data, idx_train, idx_val, idx_test, DATASET_DIR)

    print(f"\nSaved:")
    print(f"  {npz_path}")
    print(f"  {stats_path}")
    print(f"  {split_path}")
    print(f"  {seq_path}")

    return {
        "npz_path": npz_path,
        "stats_path": stats_path,
        "seq_path": seq_path
    }



# NORMALIZATION Functions

def normalize_X_dp(X_raw, EP_MAX):  # Normalizes the 10 Dp branch input features
    X = X_raw.copy().astype(np.float64)
    X[:, :6] = X[:, :6] / STRESS_SCALE
    X[:,  6] = np.log10(np.abs(X[:, 6]) + 1e-40) - SDOT_LOG_SHIFT
    X[:,  7] = X[:,  7] - LOGRHO_SHIFT
    X[:,  8] = 2.0 * X[:, 8] - 1.0
    X[:,  9] = X[:,  9] / (EP_MAX + 1e-40)
    return X.astype(np.float32)


def normalize_X_rho(X_raw, EP_MAX, dt_log_mean, dt_log_std): # Normalizes the 11 Rho branch input features.
    X = X_raw.copy().astype(np.float64)
    X[:, :6] = X[:, :6] / STRESS_SCALE
    X[:,  6] = np.log10(np.abs(X[:, 6]) + 1e-40) - SDOT_LOG_SHIFT
    X[:,  7] = X[:,  7] - LOGRHO_SHIFT
    X[:,  8] = 2.0 * X[:, 8] - 1.0
    X[:,  9] = X[:,  9] / (EP_MAX + 1e-40)

    dt_log = np.log10(np.maximum(X[:, 10], 1e-40))
    X[:, 10] = (dt_log - dt_log_mean) / (dt_log_std + 1e-40)
    return X.astype(np.float32)


def normalize_Y_dp(Y_raw, Y_std): # Divides each Dp output component by its standard deviation in training data
    return (Y_raw / (Y_std + 1e-40)).astype(np.float32)


def denormalize_Y_dp(Y_norm, Y_std): # Reverses normalization to get physical units back
    return Y_norm * Y_std


def normalize_Y_rho(Y_raw, Y_std): # Divides density rate by its standard deviation
    return (Y_raw / (Y_std + 1e-40)).astype(np.float32)


def denormalize_Y_rho(Y_norm, Y_std):
    return Y_norm * Y_std

# MODELS

class MLPRegressor(nn.Module):
  """
    A simple fully-connected (dense) neural network for regression.

    Architecture:
        Input → [Linear → (BatchNorm) → Activation → (Dropout)] × n_layers → Output

    Used for both DpNet (10 inputs, 5 outputs) and RhoNet (11 inputs, 1 output)
    """
    def __init__(self, in_dim, out_dim, hidden_dims, activation="SiLU",
                 dropout=0.0, use_bn=False):
        super().__init__()

        act_fn = getattr(nn, activation)
        layers = []
        prev = in_dim

        for h in hidden_dims:
            layers.append(nn.Linear(prev, h)) # Fully connected layer
            if use_bn:
                layers.append(nn.BatchNorm1d(h))
            layers.append(act_fn()) # Nonlinear activation
            if dropout > 0:  # Random neuron dropout
                layers.append(nn.Dropout(dropout))
            prev = h

        layers.append(nn.Linear(prev, out_dim))
        self.net = nn.Sequential(*layers)

        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)

# LOSS FUNCTIONS

class WeightedHuberLoss(nn.Module):
  """
    Huber loss weighted by the inverse standard deviation of each output component.

    We use this as some Dp components (like shear terms) have much smaller
    magnitudes than normal components. Without weighting, the loss would be
    dominated by the large components, causing poor prediction of small ones.

    Weighting by 1/std gives roughly equal importance to all components.
    Huber loss is used instead of MSE because it is less sensitive to outliers
    """
    def __init__(self, Y_std, delta=1.0):
        super().__init__()
        Y_std = np.asarray(Y_std, dtype=np.float32).reshape(-1)
        w = 1.0 / (Y_std + 1e-40)
        w = w / w.sum() * len(Y_std)
        self.register_buffer("weights", torch.tensor(w, dtype=torch.float32))
        self.huber = nn.HuberLoss(reduction="none", delta=delta)

    def forward(self, pred, target): #  Computing element-wise Huber loss and apply per-component weights
        return (self.huber(pred, target) * self.weights).mean()


class ScalarHuberLoss(nn.Module):
    def __init__(self, delta=1.0):
        super().__init__()
        self.huber = nn.HuberLoss(reduction="mean", delta=delta)

    def forward(self, pred, target):
        return self.huber(pred, target)

# DATASETS

class NoisyDataset(Dataset):
  """
    Standard point-wise dataset with optional input noise.

    Adding small random noise to inputs during training prevents the network from memorizing and improves
    generalization to slightly different test conditions
    """
    def __init__(self, X, Y, noise_std=0.01, no_noise_indices=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)
        self.noise_std = noise_std
        self.no_noise_indices = [] if no_noise_indices is None else no_noise_indices

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx].clone()
        y = self.Y[idx]

        if self.noise_std > 0:
            noise = torch.randn_like(x) * self.noise_std
            for j in self.no_noise_indices:
                noise[j] = 0.0
            x = x + noise

        return x, y


class DpRolloutWindowDataset(Dataset):
  """
    Dataset of short time-series windows for rollout training of DpNet.

    For each simulation run, we extract overlapping windows of length WINDOW_LEN.

    During training, the model predicts Dp at each step, integrates ep_cum
    forward, and is penalized for deviation from the true ep_cum trajectory.
    This teaches the model to be stable when used autoregressively.
    """

    def __init__(self, seq_csv_path, split="train", window_len=24, stride=12,
                 max_windows_per_run=None):
        self.window_len = int(window_len)
        self.stride = int(stride)

        df = pd.read_csv(seq_csv_path)
        df = df[df["split"] == split].copy()

        self.windows = []

        for stem, g in df.groupby("stem"):
            g = g.sort_values("k").reset_index(drop=True)
            n = len(g)
            count_for_run = 0
            # # Slide window across the run with given stride
            for start in range(0, n - self.window_len + 1, self.stride):
                end = start + self.window_len
                sub = g.iloc[start:end]

                item = {
                    "X_path_raw": np.column_stack([
                        sub["s_xx"].values,
                        sub["s_yy"].values,         # normalization applied inside rollout function
                        sub["s_zz"].values,
                        sub["s_yz"].values,
                        sub["s_zx"].values,
                        sub["s_xy"].values,
                        sub["sigma_dot"].values,
                        sub["log10rho_true"].values,
                        sub["loading"].values,
                        sub["ep_eq_cum"].values,
                    ]).astype(np.float32),
                    "dt": sub["dt"].values.astype(np.float32),
                    "ep_true": sub["ep_eq_cum"].values.astype(np.float32),
                    "Dp_true_5": np.column_stack([
                        sub["Dp_xx"].values,
                        sub["Dp_yy"].values,
                        sub["Dp_yz"].values,
                        sub["Dp_xz"].values,
                        sub["Dp_xy"].values,
                    ]).astype(np.float32),
                }

                self.windows.append(item)
                count_for_run += 1

                if max_windows_per_run is not None and count_for_run >= max_windows_per_run:
                    break

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        item = self.windows[idx]
        return (
            torch.tensor(item["X_path_raw"], dtype=torch.float32),
            torch.tensor(item["dt"], dtype=torch.float32),
            torch.tensor(item["ep_true"], dtype=torch.float32),
            torch.tensor(item["Dp_true_5"], dtype=torch.float32),
        )


class RhoRolloutWindowDataset(Dataset):  # Dataset of short time-series windows for rollout training of RhoNet.


    def __init__(self, seq_csv_path, split="train", window_len=24, stride=12,
                 max_windows_per_run=None):
        self.window_len = int(window_len)
        self.stride = int(stride)

        df = pd.read_csv(seq_csv_path)
        df = df[df["split"] == split].copy()

        self.windows = []

        for stem, g in df.groupby("stem"):
            g = g.sort_values("k").reset_index(drop=True)
            n = len(g)
            count_for_run = 0

            for start in range(0, n - self.window_len + 1, self.stride):
                end = start + self.window_len
                sub = g.iloc[start:end]

                item = {
                    "X_path_raw": np.column_stack([
                        sub["s_xx"].values,
                        sub["s_yy"].values,
                        sub["s_zz"].values,
                        sub["s_yz"].values,
                        sub["s_zx"].values,
                        sub["s_xy"].values,
                        sub["sigma_dot"].values,
                        sub["log10rho_true"].values,
                        sub["loading"].values,
                        sub["ep_eq_cum"].values,
                        sub["dt"].values,
                    ]).astype(np.float32),
                    "dt": sub["dt"].values.astype(np.float32),
                    "log10rho_true": sub["log10rho_true"].values.astype(np.float32),
                    "rho_rate_true": sub["dlog10rho_rate_true"].values.astype(np.float32),
                }

                self.windows.append(item)
                count_for_run += 1

                if max_windows_per_run is not None and count_for_run >= max_windows_per_run:
                    break

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        item = self.windows[idx]
        return (
            torch.tensor(item["X_path_raw"], dtype=torch.float32),
            torch.tensor(item["dt"], dtype=torch.float32),
            torch.tensor(item["log10rho_true"], dtype=torch.float32),
            torch.tensor(item["rho_rate_true"], dtype=torch.float32),
        )


def make_val_loader(X, Y, batch_size):
    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(Y, dtype=torch.float32),
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)


# TRAINING HELPERS: Dp
def run_epoch_train_one_step(model, loader, loss_fn, optimizer, device):
  """
    Runs one training epoch using only the one-step loss with no rollout.
    For each batch: forward pass → compute loss → backward pass → update weights.
    Returns average loss over all batches.
    """
    model.train()
    total = 0.0
    n = 0

    for Xb, Yb in loader:
        Xb, Yb = Xb.to(device), Yb.to(device)
        pred = model(Xb)
        loss = loss_fn(pred, Yb)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()

        total += loss.item()
        n += 1

    return total / max(n, 1)


def run_epoch_eval_one_step(model, loader, loss_fn, device):
    model.eval()
    total = 0.0
    n = 0

    with torch.no_grad():
        for Xb, Yb in loader:
            Xb, Yb = Xb.to(device), Yb.to(device)
            pred = model(Xb)
            total += loss_fn(pred, Yb).item()
            n += 1

    return total / max(n, 1)


def rollout_window_losses_dp(model, batch_X_path_raw, batch_dt, batch_ep_true,
                             batch_Dp_true_5, EP_MAX, Y_dp_std, device):
  # Computes both the one-step and rollout losses over a batch of windows
  # This makes the model to make predictions that remain accurate when composed over many steps
    B, T, _ = batch_X_path_raw.shape

    X_path_raw = batch_X_path_raw.to(device)
    dt = batch_dt.to(device)
    ep_true = batch_ep_true.to(device)
    Dp_true_5 = batch_Dp_true_5.to(device)

    Y_dp_std_t = torch.tensor(Y_dp_std, dtype=torch.float32, device=device)
    # One-step loss
    X_flat = X_path_raw.reshape(B*T, -1)
    X_flat_np = X_flat.detach().cpu().numpy()
    X_flat_norm = normalize_X_dp(X_flat_np, EP_MAX)
    X_flat_norm = torch.tensor(X_flat_norm, dtype=torch.float32, device=device)

    pred5_norm_flat = model(X_flat_norm)
    pred5_phys_flat = pred5_norm_flat * Y_dp_std_t[None, :]

    step_loss = nn.HuberLoss(reduction="mean", delta=HUBER_DELTA)(
        pred5_phys_flat, Dp_true_5.reshape(B*T, 5)
    )
    # Rollout loss: integrate ep_cum forward using predicted Dp
    ep_pred = torch.zeros((B, T), dtype=torch.float32, device=device)
    ep_pred[:, 0] = ep_true[:, 0]

    for t in range(T):
        x_t_raw = X_path_raw[:, t, :].clone()
        x_t_raw[:, 9] = ep_pred[:, t]

        x_t_np = x_t_raw.detach().cpu().numpy()
        x_t_norm = normalize_X_dp(x_t_np, EP_MAX)
        x_t_norm = torch.tensor(x_t_norm, dtype=torch.float32, device=device)

        pred5_norm_t = model(x_t_norm)
        pred5_phys_t = pred5_norm_t * Y_dp_std_t[None, :]
        epdot_t = dp5_to_epdot_eq_torch(pred5_phys_t)
        # # Euler integration: ep(t+1) = ep(t) + epdot * dt
        if t + 1 < T:
            ep_pred[:, t + 1] = ep_pred[:, t] + epdot_t * dt[:, t]

    roll_loss = nn.HuberLoss(reduction="mean", delta=HUBER_DELTA)(ep_pred, ep_true)

    return step_loss, roll_loss


def run_epoch_train_dp(model, step_loader, roll_loader, optimizer, device, EP_MAX, Y_dp_std):
  """
    Trains DpNet for one epoch using combined one-step + rollout loss.
    """
    model.train()
    step_loss_fn = WeightedHuberLoss(Y_dp_std, delta=HUBER_DELTA).to(device)

    total_combined = 0.0
    total_step = 0.0
    total_roll = 0.0
    n_batches = 0

    roll_iter = iter(roll_loader)

    for Xb, Yb in step_loader:
        Xb, Yb = Xb.to(device), Yb.to(device)

        pred_step = model(Xb)
        loss_step_main = step_loss_fn(pred_step, Yb)

        try:
            batch_roll = next(roll_iter)
        except StopIteration:
            roll_iter = iter(roll_loader)
            batch_roll = next(roll_iter)

        X_path_raw, dt, ep_true, Dp_true_5 = batch_roll
        loss_step_roll, loss_roll = rollout_window_losses_dp(
            model=model,
            batch_X_path_raw=X_path_raw,
            batch_dt=dt,
            batch_ep_true=ep_true,
            batch_Dp_true_5=Dp_true_5,
            EP_MAX=EP_MAX,
            Y_dp_std=Y_dp_std,
            device=device,
        )
        # Combined loss (rollout weighted at 20% of one-step)
        combined = W_STEP_DP * loss_step_main + 0.20 * (W_STEP_DP * loss_step_roll + W_ROLL_DP * loss_roll)

        optimizer.zero_grad()
        combined.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()

        total_combined += combined.item()
        total_step += loss_step_main.item()
        total_roll += loss_roll.item()
        n_batches += 1

    return (
        total_combined / max(n_batches, 1),
        total_step / max(n_batches, 1),
        total_roll / max(n_batches, 1),
    )


def run_epoch_eval_dp(model, val_step_loader, val_roll_loader, device, EP_MAX, Y_dp_std):
    model.eval()
    step_loss_fn = WeightedHuberLoss(Y_dp_std, delta=HUBER_DELTA).to(device)

    total_combined = 0.0
    total_step = 0.0
    total_roll = 0.0
    n_batches = 0

    roll_iter = iter(val_roll_loader)

    with torch.no_grad():
        for Xb, Yb in val_step_loader:
            Xb, Yb = Xb.to(device), Yb.to(device)

            pred_step = model(Xb)
            loss_step_main = step_loss_fn(pred_step, Yb)

            try:
                batch_roll = next(roll_iter)
            except StopIteration:
                roll_iter = iter(val_roll_loader)
                batch_roll = next(roll_iter)

            X_path_raw, dt, ep_true, Dp_true_5 = batch_roll
            loss_step_roll, loss_roll = rollout_window_losses_dp(
                model=model,
                batch_X_path_raw=X_path_raw,
                batch_dt=dt,
                batch_ep_true=ep_true,
                batch_Dp_true_5=Dp_true_5,
                EP_MAX=EP_MAX,
                Y_dp_std=Y_dp_std,
                device=device,
            )

            combined = W_STEP_DP * loss_step_main + 0.20 * (W_STEP_DP * loss_step_roll + W_ROLL_DP * loss_roll)

            total_combined += combined.item()
            total_step += loss_step_main.item()
            total_roll += loss_roll.item()
            n_batches += 1

    return (
        total_combined / max(n_batches, 1),
        total_step / max(n_batches, 1),
        total_roll / max(n_batches, 1),
    )


def train_dp_model(model, step_train_loader, step_val_loader,
                   roll_train_loader, roll_val_loader,
                   EP_MAX, Y_dp_std, out_path):
  """
    Full training loop for DpNet with:
    - Cosine annealing with Learning rate scheduler
    - Early stopping if validation loss doesn't improve for ES_PATIENCE epochs
    - Best model checkpoint saved to out_path

    Returns the trained model, training history, and best validation loss.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=LR_INIT, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=T0, T_mult=T_MULT, eta_min=LR_MIN_COSINE
    )

    print(f"\nTraining DpNet_v7 (unchanged Dp branch) up to {MAX_EPOCHS} epochs ...")

    history = dict(
        train=[], val=[], lr=[], best_epoch=0,
        train_step=[], val_step=[],
        train_roll=[], val_roll=[],
    )

    best_val = float("inf")
    best_state = None
    es_counter = 0
    t0_wall = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        tr, tr_step, tr_roll = run_epoch_train_dp(
            model=model,
            step_loader=step_train_loader,
            roll_loader=roll_train_loader,
            optimizer=optimizer,
            device=DEVICE,
            EP_MAX=EP_MAX,
            Y_dp_std=Y_dp_std,
        )

        va, va_step, va_roll = run_epoch_eval_dp(
            model=model,
            val_step_loader=step_val_loader,
            val_roll_loader=roll_val_loader,
            device=DEVICE,
            EP_MAX=EP_MAX,
            Y_dp_std=Y_dp_std,
        )

        scheduler.step(epoch - 1)
        lr = optimizer.param_groups[0]["lr"]

        history["train"].append(tr)
        history["val"].append(va)
        history["lr"].append(lr)
        history["train_step"].append(tr_step)
        history["val_step"].append(va_step)
        history["train_roll"].append(tr_roll)
        history["val_roll"].append(va_roll)

        if va < best_val:
            best_val = va
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            history["best_epoch"] = epoch
            es_counter = 0
        else:
            es_counter += 1

        if epoch % 50 == 0 or epoch == 1:
            print(
                f"  Epoch {epoch:5d} | train={tr:.5f} val={va:.5f} "
                f"| step(tr/va)=({tr_step:.5f}/{va_step:.5f}) "
                f"| roll(tr/va)=({tr_roll:.5f}/{va_roll:.5f}) "
                f"| best={best_val:.5f} | LR={lr:.2e} | ES={es_counter}/{ES_PATIENCE} | {time.time()-t0_wall:.0f}s"
            )

        if es_counter >= ES_PATIENCE:
            print(f"\nEarly stop for DpNet_v7 at epoch {epoch}  (best val={best_val:.5f} @ epoch {history['best_epoch']})")
            break

    model.load_state_dict(best_state)
    print(f"\nRestored best DpNet_v7 model from epoch {history['best_epoch']}")
    torch.save(best_state, out_path)
    print(f"Saved DpNet_v7 weights → {out_path}")

    return model, history, best_val


# TRAINING HELPERS: Rho
def rho_rollout_window_losses(model, batch_X_path_raw, batch_dt, batch_log10rho_true,
                              batch_rho_rate_true, EP_MAX, Y_rho_std,
                              dt_log_mean, dt_log_std, device):

    B, T, _ = batch_X_path_raw.shape

    X_path_raw = batch_X_path_raw.to(device)
    dt = batch_dt.to(device)
    log10rho_true = batch_log10rho_true.to(device)
    rho_rate_true = batch_rho_rate_true.to(device)

    y_std_t = torch.tensor(Y_rho_std, dtype=torch.float32, device=device)

    # one-step rate branch
    X_flat = X_path_raw.reshape(B*T, -1)
    X_flat_np = X_flat.detach().cpu().numpy()
    X_flat_norm = normalize_X_rho(X_flat_np, EP_MAX, dt_log_mean, dt_log_std)
    X_flat_norm = torch.tensor(X_flat_norm, dtype=torch.float32, device=device)

    pred_rate_norm_flat = model(X_flat_norm).reshape(B, T)
    pred_rate_phys_flat = pred_rate_norm_flat * y_std_t

    step_loss = nn.HuberLoss(reduction="mean", delta=HUBER_DELTA)(
        pred_rate_phys_flat, rho_rate_true
    )

    # rollout branch on log10rho
    log10rho_pred = torch.zeros((B, T), dtype=torch.float32, device=device)
    log10rho_pred[:, 0] = log10rho_true[:, 0]

    for t in range(T):
        x_t_raw = X_path_raw[:, t, :].clone()
        x_t_raw[:, 7] = log10rho_pred[:, t]   # recursive predicted log10rho

        x_t_np = x_t_raw.detach().cpu().numpy()
        x_t_norm = normalize_X_rho(x_t_np, EP_MAX, dt_log_mean, dt_log_std)
        x_t_norm = torch.tensor(x_t_norm, dtype=torch.float32, device=device)

        pred_rate_norm_t = model(x_t_norm).squeeze(-1)
        pred_rate_phys_t = pred_rate_norm_t * y_std_t

        if t + 1 < T:
            log10rho_pred[:, t + 1] = log10rho_pred[:, t] + pred_rate_phys_t * dt[:, t]

    roll_loss = nn.HuberLoss(reduction="mean", delta=HUBER_DELTA)(
        log10rho_pred, log10rho_true
    )

    return step_loss, roll_loss


def run_epoch_train_rho(model, step_loader, roll_loader, optimizer, device,
                        EP_MAX, Y_rho_std, dt_log_mean, dt_log_std):
    model.train()
    step_loss_fn = ScalarHuberLoss(delta=HUBER_DELTA).to(device)

    total_combined = 0.0
    total_step = 0.0
    total_roll = 0.0
    n_batches = 0

    roll_iter = iter(roll_loader)

    for Xb, Yb in step_loader:
        Xb, Yb = Xb.to(device), Yb.to(device)

        pred_step = model(Xb)
        loss_step_main = step_loss_fn(pred_step, Yb)

        try:
            batch_roll = next(roll_iter)
        except StopIteration:
            roll_iter = iter(roll_loader)
            batch_roll = next(roll_iter)

        X_path_raw, dt, log10rho_true, rho_rate_true = batch_roll
        loss_step_roll, loss_roll = rho_rollout_window_losses(
            model=model,
            batch_X_path_raw=X_path_raw,
            batch_dt=dt,
            batch_log10rho_true=log10rho_true,
            batch_rho_rate_true=rho_rate_true,
            EP_MAX=EP_MAX,
            Y_rho_std=Y_rho_std,
            dt_log_mean=dt_log_mean,
            dt_log_std=dt_log_std,
            device=device,
        )

        combined = W_STEP_RHO * loss_step_main + 0.30 * (W_STEP_RHO * loss_step_roll + W_ROLL_RHO * loss_roll)

        optimizer.zero_grad()
        combined.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()

        total_combined += combined.item()
        total_step += loss_step_main.item()
        total_roll += loss_roll.item()
        n_batches += 1

    return (
        total_combined / max(n_batches, 1),
        total_step / max(n_batches, 1),
        total_roll / max(n_batches, 1),
    )


def run_epoch_eval_rho(model, val_step_loader, val_roll_loader, device,
                       EP_MAX, Y_rho_std, dt_log_mean, dt_log_std):
    model.eval()
    step_loss_fn = ScalarHuberLoss(delta=HUBER_DELTA).to(device)

    total_combined = 0.0
    total_step = 0.0
    total_roll = 0.0
    n_batches = 0

    roll_iter = iter(val_roll_loader)

    with torch.no_grad():
        for Xb, Yb in val_step_loader:
            Xb, Yb = Xb.to(device), Yb.to(device)

            pred_step = model(Xb)
            loss_step_main = step_loss_fn(pred_step, Yb)

            try:
                batch_roll = next(roll_iter)
            except StopIteration:
                roll_iter = iter(val_roll_loader)
                batch_roll = next(roll_iter)

            X_path_raw, dt, log10rho_true, rho_rate_true = batch_roll
            loss_step_roll, loss_roll = rho_rollout_window_losses(
                model=model,
                batch_X_path_raw=X_path_raw,
                batch_dt=dt,
                batch_log10rho_true=log10rho_true,
                batch_rho_rate_true=rho_rate_true,
                EP_MAX=EP_MAX,
                Y_rho_std=Y_rho_std,
                dt_log_mean=dt_log_mean,
                dt_log_std=dt_log_std,
                device=device,
            )

            combined = W_STEP_RHO * loss_step_main + 0.30 * (W_STEP_RHO * loss_step_roll + W_ROLL_RHO * loss_roll)

            total_combined += combined.item()
            total_step += loss_step_main.item()
            total_roll += loss_roll.item()
            n_batches += 1

    return (
        total_combined / max(n_batches, 1),
        total_step / max(n_batches, 1),
        total_roll / max(n_batches, 1),
    )


def train_rho_model(model, step_train_loader, step_val_loader,
                    roll_train_loader, roll_val_loader,
                    EP_MAX, Y_rho_std, dt_log_mean, dt_log_std, out_path):
    optimizer = torch.optim.Adam(model.parameters(), lr=LR_INIT, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=T0, T_mult=T_MULT, eta_min=LR_MIN_COSINE
    )

    print(f"\nTraining RhoNet_v7 (improved rho branch) up to {MAX_EPOCHS} epochs ...")

    history = dict(
        train=[], val=[], lr=[], best_epoch=0,
        train_step=[], val_step=[],
        train_roll=[], val_roll=[],
    )

    best_val = float("inf")
    best_state = None
    es_counter = 0
    t0_wall = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        tr, tr_step, tr_roll = run_epoch_train_rho(
            model=model,
            step_loader=step_train_loader,
            roll_loader=roll_train_loader,
            optimizer=optimizer,
            device=DEVICE,
            EP_MAX=EP_MAX,
            Y_rho_std=Y_rho_std,
            dt_log_mean=dt_log_mean,
            dt_log_std=dt_log_std,
        )

        va, va_step, va_roll = run_epoch_eval_rho(
            model=model,
            val_step_loader=step_val_loader,
            val_roll_loader=roll_val_loader,
            device=DEVICE,
            EP_MAX=EP_MAX,
            Y_rho_std=Y_rho_std,
            dt_log_mean=dt_log_mean,
            dt_log_std=dt_log_std,
        )

        scheduler.step(epoch - 1)
        lr = optimizer.param_groups[0]["lr"]

        history["train"].append(tr)
        history["val"].append(va)
        history["lr"].append(lr)
        history["train_step"].append(tr_step)
        history["val_step"].append(va_step)
        history["train_roll"].append(tr_roll)
        history["val_roll"].append(va_roll)

        if va < best_val:
            best_val = va
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            history["best_epoch"] = epoch
            es_counter = 0
        else:
            es_counter += 1

        if epoch % 50 == 0 or epoch == 1:
            print(
                f"  Epoch {epoch:5d} | train={tr:.5f} val={va:.5f} "
                f"| step(tr/va)=({tr_step:.5f}/{va_step:.5f}) "
                f"| roll(tr/va)=({tr_roll:.5f}/{va_roll:.5f}) "
                f"| best={best_val:.5f} | LR={lr:.2e} | ES={es_counter}/{ES_PATIENCE} | {time.time()-t0_wall:.0f}s"
            )

        if es_counter >= ES_PATIENCE:
            print(f"\nEarly stop for RhoNet_v7 at epoch {epoch}  (best val={best_val:.5f} @ epoch {history['best_epoch']})")
            break

    model.load_state_dict(best_state)
    print(f"\nRestored best RhoNet_v7 model from epoch {history['best_epoch']}")
    torch.save(best_state, out_path)
    print(f"Saved RhoNet_v7 weights → {out_path}")

    return model, history, best_val



# METRICS

def evaluate_dp_model(model, X, Y_raw5, Y_std5, device):
    model.eval()
    with torch.no_grad():
        pred5_norm = model(torch.tensor(X, dtype=torch.float32).to(device)).cpu().numpy()

    pred5_phys = denormalize_Y_dp(pred5_norm, Y_std5)
    pred6_phys = reconstruct_Y6_from_Y5(pred5_phys)

    true6 = np.column_stack([
        Y_raw5[:, 0],
        Y_raw5[:, 1],
        -(Y_raw5[:, 0] + Y_raw5[:, 1]),
        Y_raw5[:, 2],
        Y_raw5[:, 3],
        Y_raw5[:, 4],
    ])

    r2 = np.zeros(6)
    rmse = np.zeros(6)

    for i in range(6):
        ss_res = np.sum((true6[:, i] - pred6_phys[:, i]) ** 2)
        ss_tot = np.sum((true6[:, i] - true6[:, i].mean()) ** 2)
        r2[i] = 1.0 - ss_res / (ss_tot + 1e-40)
        rmse[i] = np.sqrt(np.mean((true6[:, i] - pred6_phys[:, i]) ** 2))

    return r2, rmse, pred6_phys, true6


def evaluate_rho_model(model, X_rho_norm, Y_rho_raw_rate, Y_rho_std, device, seq_csv_path):

    model.eval()
    with torch.no_grad():
        pred_norm = model(torch.tensor(X_rho_norm, dtype=torch.float32).to(device)).cpu().numpy()

    pred_phys = denormalize_Y_rho(pred_norm, Y_rho_std)
    true = Y_rho_raw_rate.copy()

    ss_res = np.sum((true[:, 0] - pred_phys[:, 0]) ** 2)
    ss_tot = np.sum((true[:, 0] - true[:, 0].mean()) ** 2)
    r2 = 1.0 - ss_res / (ss_tot + 1e-40)
    rmse = np.sqrt(np.mean((true[:, 0] - pred_phys[:, 0]) ** 2))

    return r2, rmse, pred_phys, true

# INFERENCE HELPERS

def predict_dp_once(dp_model, x_raw, EP_MAX, Y_dp_std):
    x_norm = normalize_X_dp(x_raw, EP_MAX)
    with torch.no_grad():
        pred5_norm = dp_model(torch.tensor(x_norm, dtype=torch.float32).to(DEVICE)).cpu().numpy()
    pred5_phys = denormalize_Y_dp(pred5_norm, Y_dp_std)
    pred6_phys = reconstruct_Y6_from_Y5(pred5_phys)
    return pred6_phys[0]


def predict_rho_rate_once(rho_model, x_raw, EP_MAX, Y_rho_std, dt_log_mean, dt_log_std):
    x_norm = normalize_X_rho(x_raw, EP_MAX, dt_log_mean, dt_log_std)
    with torch.no_grad():
        pred_norm = rho_model(torch.tensor(x_norm, dtype=torch.float32).to(DEVICE)).cpu().numpy()
    pred_phys = denormalize_Y_rho(pred_norm, Y_rho_std)
    return float(pred_phys[0, 0])

# ROLLOUT EVALUATION

def rollout_one_run_modeA(dp_model, run_df, EP_MAX, Y_dp_std):

    run_df = run_df.sort_values("k").reset_index(drop=True).copy()

    ep_true = run_df["ep_eq_cum"].values.astype(np.float64)
    log10rho_true = run_df["log10rho_true"].values.astype(np.float64)

    ep_pred = np.zeros(len(run_df), dtype=np.float64)
    ep_pred[0] = ep_true[0]

    rows = []

    for i in range(len(run_df)):
        row = run_df.iloc[i]

        x_raw = np.array([[
            row["s_xx"], row["s_yy"], row["s_zz"],
            row["s_yz"], row["s_zx"], row["s_xy"],
            row["sigma_dot"], log10rho_true[i], row["loading"], ep_pred[i]
        ]], dtype=np.float64)

        pred6 = predict_dp_once(dp_model, x_raw, EP_MAX, Y_dp_std)
        epdot_eq = dp6_to_epdot_eq_np(pred6)

        rows.append({
            "stem": row["stem"],
            "k": int(row["k"]),
            "time": float(row["time"]),
            "dt": float(row["dt"]),
            "s_xx": float(row["s_xx"]),
            "s_yy": float(row["s_yy"]),
            "s_zz": float(row["s_zz"]),
            "s_yz": float(row["s_yz"]),
            "s_zx": float(row["s_zx"]),
            "s_xy": float(row["s_xy"]),
            "ep_eq_true": float(ep_true[i]),
            "ep_eq_pred": float(ep_pred[i]),
            "epdot_eq_pred": float(epdot_eq),
            "log10rho_true": float(log10rho_true[i]),
            "rho_true": float(10.0 ** log10rho_true[i]),
            "Dp_xx_pred": float(pred6[0]),
            "Dp_yy_pred": float(pred6[1]),
            "Dp_zz_pred": float(pred6[2]),
            "Dp_yz_pred": float(pred6[3]),
            "Dp_xz_pred": float(pred6[4]),
            "Dp_xy_pred": float(pred6[5]),
            "Dp_xx_true": float(row["Dp_xx"]),
            "Dp_yy_true": float(row["Dp_yy"]),
            "Dp_zz_true": float(row["Dp_zz"]),
            "Dp_yz_true": float(row["Dp_yz"]),
            "Dp_xz_true": float(row["Dp_xz"]),
            "Dp_xy_true": float(row["Dp_xy"]),
        })

        if i + 1 < len(run_df):
            dt = float(run_df.iloc[i]["dt"])
            ep_pred[i + 1] = ep_pred[i] + epdot_eq * dt

    return pd.DataFrame(rows)


def rollout_one_run_modeB(dp_model, rho_model, run_df, EP_MAX, Y_dp_std, Y_rho_std, dt_log_mean, dt_log_std):

    run_df = run_df.sort_values("k").reset_index(drop=True).copy()

    ep_true = run_df["ep_eq_cum"].values.astype(np.float64)
    log10rho_true = run_df["log10rho_true"].values.astype(np.float64)
    dt_arr = run_df["dt"].values.astype(np.float64)

    ep_pred = np.zeros(len(run_df), dtype=np.float64)
    log10rho_pred = np.zeros(len(run_df), dtype=np.float64)

    ep_pred[0] = ep_true[0]
    log10rho_pred[0] = log10rho_true[0]

    rows = []

    for i in range(len(run_df)):
        row = run_df.iloc[i]

        x_dp_raw = np.array([[
            row["s_xx"], row["s_yy"], row["s_zz"],
            row["s_yz"], row["s_zx"], row["s_xy"],
            row["sigma_dot"], log10rho_pred[i], row["loading"], ep_pred[i]
        ]], dtype=np.float64)

        pred6 = predict_dp_once(dp_model, x_dp_raw, EP_MAX, Y_dp_std)
        epdot_eq = dp6_to_epdot_eq_np(pred6)

        x_rho_raw = np.array([[
            row["s_xx"], row["s_yy"], row["s_zz"],
            row["s_yz"], row["s_zx"], row["s_xy"],
            row["sigma_dot"], log10rho_pred[i], row["loading"], ep_pred[i], row["dt"]
        ]], dtype=np.float64)

        rho_rate_pred = predict_rho_rate_once(
            rho_model, x_rho_raw, EP_MAX, Y_rho_std, dt_log_mean, dt_log_std
        )

        rows.append({
            "stem": row["stem"],
            "k": int(row["k"]),
            "time": float(row["time"]),
            "dt": float(row["dt"]),
            "s_xx": float(row["s_xx"]),
            "s_yy": float(row["s_yy"]),
            "s_zz": float(row["s_zz"]),
            "s_yz": float(row["s_yz"]),
            "s_zx": float(row["s_zx"]),
            "s_xy": float(row["s_xy"]),
            "ep_eq_true": float(ep_true[i]),
            "ep_eq_pred": float(ep_pred[i]),
            "epdot_eq_pred": float(epdot_eq),
            "log10rho_true": float(log10rho_true[i]),
            "log10rho_pred": float(log10rho_pred[i]),
            "rho_true": float(10.0 ** log10rho_true[i]),
            "rho_pred": float(10.0 ** log10rho_pred[i]),
            "dlog10rho_true": float(row["dlog10rho_true"]),
            "dlog10rho_rate_true": float(row["dlog10rho_rate_true"]),
            "rho_rate_pred": float(rho_rate_pred),
            "Dp_xx_pred": float(pred6[0]),
            "Dp_yy_pred": float(pred6[1]),
            "Dp_zz_pred": float(pred6[2]),
            "Dp_yz_pred": float(pred6[3]),
            "Dp_xz_pred": float(pred6[4]),
            "Dp_xy_pred": float(pred6[5]),
            "Dp_xx_true": float(row["Dp_xx"]),
            "Dp_yy_true": float(row["Dp_yy"]),
            "Dp_zz_true": float(row["Dp_zz"]),
            "Dp_yz_true": float(row["Dp_yz"]),
            "Dp_xz_true": float(row["Dp_xz"]),
            "Dp_xy_true": float(row["Dp_xy"]),
        })

        if i + 1 < len(run_df):
            dt = dt_arr[i]
            ep_pred[i + 1] = ep_pred[i] + epdot_eq * dt
            log10rho_pred[i + 1] = log10rho_pred[i] + rho_rate_pred * dt

    return pd.DataFrame(rows)


def rollout_all_test_runs_modeA(dp_model, seq_csv_path, EP_MAX, Y_dp_std, out_dir):
    df_seq = pd.read_csv(seq_csv_path)
    test_runs = sorted(df_seq.loc[df_seq["split"] == "test", "stem"].unique())

    all_summary = []

    for stem in test_runs:
        run_df = df_seq[(df_seq["split"] == "test") & (df_seq["stem"] == stem)].copy()
        pred_df = rollout_one_run_modeA(dp_model, run_df, EP_MAX, Y_dp_std)

        ep_rel_err = np.mean(
            np.abs(pred_df["ep_eq_pred"] - pred_df["ep_eq_true"]) /
            np.maximum(np.abs(pred_df["ep_eq_true"]), 1e-30)
        )

        all_summary.append({
            "stem": stem,
            "ep_eq_rel_err": ep_rel_err
        })

        safe_stem = re.sub(r"[^A-Za-z0-9_\-]+", "_", stem)
        pred_df.to_csv(os.path.join(out_dir, f"rollout_modeA_{safe_stem}.csv"), index=False)

    summary_df = pd.DataFrame(all_summary)
    summary_path = os.path.join(out_dir, "rollout_summary_modeA.csv")
    summary_df.to_csv(summary_path, index=False)

    print("\nMode A rollout summary (pred ep_eq + TRUE density):")
    print(summary_df)
    print("\nMean rollout ep_eq error:")
    print(summary_df["ep_eq_rel_err"].mean())

    return summary_df


def rollout_all_test_runs_modeB(dp_model, rho_model, seq_csv_path, EP_MAX, Y_dp_std, Y_rho_std,
                                dt_log_mean, dt_log_std, out_dir):
    df_seq = pd.read_csv(seq_csv_path)
    test_runs = sorted(df_seq.loc[df_seq["split"] == "test", "stem"].unique())

    all_summary = []

    for stem in test_runs:
        run_df = df_seq[(df_seq["split"] == "test") & (df_seq["stem"] == stem)].copy()
        pred_df = rollout_one_run_modeB(
            dp_model, rho_model, run_df, EP_MAX, Y_dp_std, Y_rho_std, dt_log_mean, dt_log_std
        )

        ep_rel_err = np.mean(
            np.abs(pred_df["ep_eq_pred"] - pred_df["ep_eq_true"]) /
            np.maximum(np.abs(pred_df["ep_eq_true"]), 1e-30)
        )

        rho_rel_err = np.mean(
            np.abs(pred_df["rho_pred"] - pred_df["rho_true"]) /
            np.maximum(np.abs(pred_df["rho_true"]), 1e-30)
        )

        all_summary.append({
            "stem": stem,
            "ep_eq_rel_err": ep_rel_err,
            "rho_rel_err": rho_rel_err
        })

        safe_stem = re.sub(r"[^A-Za-z0-9_\-]+", "_", stem)
        pred_df.to_csv(os.path.join(out_dir, f"rollout_modeB_{safe_stem}.csv"), index=False)

    summary_df = pd.DataFrame(all_summary)
    summary_path = os.path.join(out_dir, "rollout_summary_modeB.csv")
    summary_df.to_csv(summary_path, index=False)

    print("\nMode B rollout summary (pred ep_eq + PRED density, rho-rate integration):")
    print(summary_df)
    print("\nMean rollout ep_eq error:")
    print(summary_df["ep_eq_rel_err"].mean())
    print("\nMean rollout rho error:")
    print(summary_df["rho_rel_err"].mean())

    return summary_df

# PLOTTING

def plot_training_curves(history, out_path_prefix):
    epochs = range(1, len(history["train"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].plot(epochs, history["train"], label="train")
    axes[0].plot(epochs, history["val"], label="val")
    axes[0].axvline(history["best_epoch"], color="red", ls=":", label=f"best={history['best_epoch']}")
    axes[0].set_yscale("log")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Loss curves")
    axes[0].legend()

    axes[1].plot(epochs, history["lr"])
    axes[1].set_yscale("log")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("LR")
    axes[1].set_title("Cosine annealing LR with warm restarts")

    plt.tight_layout()
    plt.savefig(out_path_prefix + "_train_curves.png", dpi=150)
    plt.close()


def plot_branch_training_detail(history, out_path, title):
    epochs = range(1, len(history["train"]) + 1)
    plt.figure(figsize=(12, 4))
    plt.plot(epochs, history["train_step"], label="train step")
    plt.plot(epochs, history["val_step"], label="val step")
    plt.plot(epochs, history["train_roll"], label="train roll")
    plt.plot(epochs, history["val_roll"], label="val roll")
    plt.yscale("log")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_dp_per_component(r2s, rmses, out_dir):
    x = np.arange(6)
    w = 0.25

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for j, split in enumerate(["train", "val", "test"]):
        axes[0].bar(x + (j - 1) * w, r2s[split], w, label=split)
        axes[1].bar(x + (j - 1) * w, rmses[split], w, label=split)

    axes[0].axhline(1.0, color="k", ls="--", lw=0.8)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(DP_OUTPUT_LABELS_6)
    axes[0].set_ylabel("R²")
    axes[0].set_title("Dp model: R² per component")
    axes[0].legend()

    axes[1].set_xticks(x)
    axes[1].set_xticklabels(DP_OUTPUT_LABELS_6)
    axes[1].set_ylabel("RMSE")
    axes[1].set_title("Dp model: RMSE per component")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "dp_per_component_metrics.png"), dpi=150)
    plt.close()


def plot_rollout_stress_vs_plastic_strain(pattern_prefix, out_dir, max_plots=None):
    import glob

    csv_files = sorted(glob.glob(os.path.join(out_dir, f"{pattern_prefix}_*.csv")))
    csv_files = [f for f in csv_files if "summary" not in os.path.basename(f).lower()]

    if not csv_files:
        print(f"No rollout CSV files found for {pattern_prefix}.")
        return

    if max_plots is not None:
        csv_files = csv_files[:max_plots]

    n = len(csv_files)
    ncols = 2
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4.5 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for ax, fpath in zip(axes, csv_files):
        df = pd.read_csv(fpath)
        stem = str(df["stem"].iloc[0])

        if "uniaxial" in stem.lower():
            stress = df["s_zz"].values / 1e6
            ylabel = r"$\sigma_{zz}$ (MPa)"
        else:
            stress = df["s_xy"].values / 1e6
            ylabel = r"$\tau_{xy}$ (MPa)"

        ep_true = df["ep_eq_true"].values * 100.0
        ep_pred = df["ep_eq_pred"].values * 100.0

        ax.plot(ep_true, stress, label="DDD true path", linewidth=2)
        ax.plot(ep_pred, stress, "--", label="NN rollout path", linewidth=2)

        ax.set_title(stem, fontsize=9)
        ax.set_xlabel(r"$\varepsilon_{eq}^{p}$ (%)")
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)

    for j in range(len(csv_files), len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()


def plot_rollout_density_vs_plastic_strain(pattern_prefix, out_dir, max_plots=None):
    import glob

    csv_files = sorted(glob.glob(os.path.join(out_dir, f"{pattern_prefix}_*.csv")))
    csv_files = [f for f in csv_files if "summary" not in os.path.basename(f).lower()]

    if not csv_files:
        print(f"No rollout CSV files found for {pattern_prefix}.")
        return

    if max_plots is not None:
        csv_files = csv_files[:max_plots]

    n = len(csv_files)
    ncols = 2
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4.5 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for ax, fpath in zip(axes, csv_files):
        df = pd.read_csv(fpath)
        stem = str(df["stem"].iloc[0])

        ep_true = df["ep_eq_true"].values * 100.0
        ax.plot(ep_true, df["rho_true"].values, label="DDD true density", linewidth=2)
        ax.plot(ep_true, df["rho_pred"].values, "--", label="NN rollout density", linewidth=2)

        ax.set_title(stem, fontsize=9)
        ax.set_xlabel(r"$\varepsilon_{eq}^{p}$ (%)")
        ax.set_ylabel(r"$\rho$ (m$^{-2}$)")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)

    for j in range(len(csv_files), len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()

# MAIN

def train_and_evaluate_v7(npz_path, stats_path, seq_path):
    print(f"Device : {DEVICE}")
    print(f"PyTorch: {torch.__version__}\n")

    print("Loading dataset v7_rho_improved ...")
    data = np.load(npz_path)
    stats = json.load(open(stats_path))

    X_train_dp_raw = data["X_train_dp_raw"]
    X_val_dp_raw   = data["X_val_dp_raw"]
    X_test_dp_raw  = data["X_test_dp_raw"]

    Y_train_dp_raw = data["Y_train_dp_raw"]
    Y_val_dp_raw   = data["Y_val_dp_raw"]
    Y_test_dp_raw  = data["Y_test_dp_raw"]

    X_train_rho_raw = data["X_train_rho_raw"]
    X_val_rho_raw   = data["X_val_rho_raw"]
    X_test_rho_raw  = data["X_test_rho_raw"]

    Y_train_rho_raw = data["Y_train_rho_raw"]
    Y_val_rho_raw   = data["Y_val_rho_raw"]
    Y_test_rho_raw  = data["Y_test_rho_raw"]

    EP_MAX      = float(stats["EP_MAX"])
    Y_dp_std    = np.array(stats["Y_dp_std"], dtype=np.float64)
    Y_rho_std   = float(stats["Y_rho_std"])
    DT_LOG_MEAN = float(stats["DT_LOG_MEAN"])
    DT_LOG_STD  = float(stats["DT_LOG_STD"])

    print(f"  Train Dp: {X_train_dp_raw.shape[0]:,} | Val Dp: {X_val_dp_raw.shape[0]:,} | Test Dp: {X_test_dp_raw.shape[0]:,}")
    print(f"  Train Rho: {X_train_rho_raw.shape[0]:,} | Val Rho: {X_val_rho_raw.shape[0]:,} | Test Rho: {X_test_rho_raw.shape[0]:,}")
    print(f"  EP_MAX: {EP_MAX:.4e}")
    print(f"  Y_dp_std: {[f'{v:.1f}' for v in Y_dp_std]}")
    print(f"  Y_rho_std: {Y_rho_std:.4e}")
    print(f"  DT_LOG_MEAN: {DT_LOG_MEAN:.4f}  DT_LOG_STD: {DT_LOG_STD:.4f}")

    # normalize Dp branch
    X_train_dp = normalize_X_dp(X_train_dp_raw, EP_MAX)
    X_val_dp   = normalize_X_dp(X_val_dp_raw,   EP_MAX)
    X_test_dp  = normalize_X_dp(X_test_dp_raw,  EP_MAX)

    Y_train_dp = normalize_Y_dp(Y_train_dp_raw, Y_dp_std)
    Y_val_dp   = normalize_Y_dp(Y_val_dp_raw,   Y_dp_std)
    Y_test_dp  = normalize_Y_dp(Y_test_dp_raw,  Y_dp_std)

    # normalize Rho branch
    X_train_rho = normalize_X_rho(X_train_rho_raw, EP_MAX, DT_LOG_MEAN, DT_LOG_STD)
    X_val_rho   = normalize_X_rho(X_val_rho_raw,   EP_MAX, DT_LOG_MEAN, DT_LOG_STD)
    X_test_rho  = normalize_X_rho(X_test_rho_raw,  EP_MAX, DT_LOG_MEAN, DT_LOG_STD)

    Y_train_rho = normalize_Y_rho(Y_train_rho_raw, Y_rho_std)
    Y_val_rho   = normalize_Y_rho(Y_val_rho_raw,   Y_rho_std)
    Y_test_rho  = normalize_Y_rho(Y_test_rho_raw,  Y_rho_std)

    print("\nNormalized input ranges (train) — Dp branch:")
    for i, label in enumerate(INPUT_LABELS_DP):
        print(f"  {label:10s}: [{X_train_dp[:,i].min():.3f}, {X_train_dp[:,i].max():.3f}]  mean={X_train_dp[:,i].mean():.3f}")

    print("\nNormalized input ranges (train) — Rho branch:")
    for i, label in enumerate(INPUT_LABELS_RHO):
        print(f"  {label:10s}: [{X_train_rho[:,i].min():.3f}, {X_train_rho[:,i].max():.3f}]  mean={X_train_rho[:,i].mean():.3f}")


    # Dp branch loaders

    dp_train_ds = NoisyDataset(X_train_dp, Y_train_dp, noise_std=INPUT_NOISE_STD, no_noise_indices=[8])
    dp_train_loader = DataLoader(
        dp_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=(DEVICE == "cuda")
    )
    dp_val_loader = make_val_loader(X_val_dp, Y_val_dp, BATCH_SIZE)

    dp_roll_train_ds = DpRolloutWindowDataset(
        seq_csv_path=seq_path, split="train", window_len=WINDOW_LEN, stride=WINDOW_STRIDE,
        max_windows_per_run=MAX_WINDOWS_PER_RUN
    )
    dp_roll_val_ds = DpRolloutWindowDataset(
        seq_csv_path=seq_path, split="val", window_len=WINDOW_LEN, stride=WINDOW_STRIDE,
        max_windows_per_run=MAX_WINDOWS_PER_RUN
    )

    dp_roll_train_loader = DataLoader(
        dp_roll_train_ds, batch_size=ROLL_BATCH_SIZE, shuffle=True, num_workers=0,
        pin_memory=(DEVICE == "cuda")
    )
    dp_roll_val_loader = DataLoader(
        dp_roll_val_ds, batch_size=ROLL_BATCH_SIZE, shuffle=False, num_workers=0
    )


    # Rho branch loader
    rho_train_ds = NoisyDataset(X_train_rho, Y_train_rho, noise_std=INPUT_NOISE_STD, no_noise_indices=[8, 10])
    rho_train_loader = DataLoader(
        rho_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=(DEVICE == "cuda")
    )
    rho_val_loader = make_val_loader(X_val_rho, Y_val_rho, BATCH_SIZE)

    rho_roll_train_ds = RhoRolloutWindowDataset(
        seq_csv_path=seq_path, split="train", window_len=WINDOW_LEN, stride=WINDOW_STRIDE,
        max_windows_per_run=MAX_WINDOWS_PER_RUN
    )
    rho_roll_val_ds = RhoRolloutWindowDataset(
        seq_csv_path=seq_path, split="val", window_len=WINDOW_LEN, stride=WINDOW_STRIDE,
        max_windows_per_run=MAX_WINDOWS_PER_RUN
    )

    rho_roll_train_loader = DataLoader(
        rho_roll_train_ds, batch_size=ROLL_BATCH_SIZE, shuffle=True, num_workers=0,
        pin_memory=(DEVICE == "cuda")
    )
    rho_roll_val_loader = DataLoader(
        rho_roll_val_ds, batch_size=ROLL_BATCH_SIZE, shuffle=False, num_workers=0
    )

    print(f"\nDp rollout windows: train={len(dp_roll_train_ds):,}, val={len(dp_roll_val_ds):,}")
    print(f"Rho rollout windows: train={len(rho_roll_train_ds):,}, val={len(rho_roll_val_ds):,}")


    # Dp model
    dp_model = MLPRegressor(
        in_dim=10,
        out_dim=5,
        hidden_dims=DP_HIDDEN_DIMS,
        activation=DP_ACTIVATION,
        dropout=DP_DROPOUT,
        use_bn=DP_USE_BATCHNORM,
    ).to(DEVICE)

    n_dp_params = sum(p.numel() for p in dp_model.parameters() if p.requires_grad)
    print(f"\nDp model: {n_dp_params:,} trainable parameters")
    print(dp_model)

    dp_weights_path = os.path.join(NN_OUT_DIR, "dpnet_v7_best_state.pt")
    dp_model, dp_history, dp_best_val = train_dp_model(
        model=dp_model,
        step_train_loader=dp_train_loader,
        step_val_loader=dp_val_loader,
        roll_train_loader=dp_roll_train_loader,
        roll_val_loader=dp_roll_val_loader,
        EP_MAX=EP_MAX,
        Y_dp_std=Y_dp_std,
        out_path=dp_weights_path,
    )

    torch.save({
        "model_state_dict": dp_model.state_dict(),
        "model_config": {
            "in_dim": 10,
            "out_dim": 5,
            "hidden_dims": DP_HIDDEN_DIMS,
            "activation": DP_ACTIVATION,
            "dropout": DP_DROPOUT,
            "use_bn": DP_USE_BATCHNORM,
        },
        "norm_config": {
            "STRESS_SCALE": STRESS_SCALE,
            "LOGRHO_SHIFT": LOGRHO_SHIFT,
            "SDOT_LOG_SHIFT": SDOT_LOG_SHIFT,
            "EP_MAX": EP_MAX,
            "Y_dp_std": Y_dp_std.tolist(),
        },
        "training_config": {
            "window_len": WINDOW_LEN,
            "window_stride": WINDOW_STRIDE,
            "roll_start_epoch": ROLL_START_EPOCH,
            "W_STEP_DP": W_STEP_DP,
            "W_ROLL_DP": W_ROLL_DP,
        },
        "best_val_loss": dp_best_val,
        "input_order": INPUT_LABELS_DP,
        "output_note": "Predicts 5 free Dp with rollout-window training. Dp_zz = -(Dp_xx+Dp_yy).",
    }, os.path.join(NN_OUT_DIR, "dpnet_v7_best.pt"))


    rho_model = MLPRegressor(
        in_dim=11,
        out_dim=1,
        hidden_dims=RHO_HIDDEN_DIMS,
        activation=RHO_ACTIVATION,
        dropout=RHO_DROPOUT,
        use_bn=RHO_USE_BATCHNORM,
    ).to(DEVICE)

    n_rho_params = sum(p.numel() for p in rho_model.parameters() if p.requires_grad)
    print(f"\nRho model: {n_rho_params:,} trainable parameters")
    print(rho_model)

    rho_weights_path = os.path.join(NN_OUT_DIR, "rhonet_v7_best_state.pt")
    rho_model, rho_history, rho_best_val = train_rho_model(
        model=rho_model,
        step_train_loader=rho_train_loader,
        step_val_loader=rho_val_loader,
        roll_train_loader=rho_roll_train_loader,
        roll_val_loader=rho_roll_val_loader,
        EP_MAX=EP_MAX,
        Y_rho_std=Y_rho_std,
        dt_log_mean=DT_LOG_MEAN,
        dt_log_std=DT_LOG_STD,
        out_path=rho_weights_path,
    )

    torch.save({
        "model_state_dict": rho_model.state_dict(),
        "model_config": {
            "in_dim": 11,
            "out_dim": 1,
            "hidden_dims": RHO_HIDDEN_DIMS,
            "activation": RHO_ACTIVATION,
            "dropout": RHO_DROPOUT,
            "use_bn": RHO_USE_BATCHNORM,
        },
        "norm_config": {
            "STRESS_SCALE": STRESS_SCALE,
            "LOGRHO_SHIFT": LOGRHO_SHIFT,
            "SDOT_LOG_SHIFT": SDOT_LOG_SHIFT,
            "EP_MAX": EP_MAX,
            "Y_rho_std": Y_rho_std,
            "DT_LOG_MEAN": DT_LOG_MEAN,
            "DT_LOG_STD": DT_LOG_STD,
        },
        "training_config": {
            "window_len": WINDOW_LEN,
            "window_stride": WINDOW_STRIDE,
            "roll_start_epoch": ROLL_START_EPOCH,
            "W_STEP_RHO": W_STEP_RHO,
            "W_ROLL_RHO": W_ROLL_RHO,
        },
        "best_val_loss": rho_best_val,
        "input_order": INPUT_LABELS_RHO,
        "output_note": "Predicts d(log10rho)/dt with rollout-window training on log10rho trajectory.",
    }, os.path.join(NN_OUT_DIR, "rhonet_v7_best.pt"))


    # One-step evaluation: Dp

    print("\nEvaluating Dp model on all splits ...")
    dp_r2s, dp_rmses = {}, {}
    dp_preds, dp_trues = {}, {}
    for split, X, Y_raw in [
        ("train", X_train_dp, Y_train_dp_raw),
        ("val",   X_val_dp,   Y_val_dp_raw),
        ("test",  X_test_dp,  Y_test_dp_raw),
    ]:
        r2, rmse, pred6, true6 = evaluate_dp_model(dp_model, X, Y_raw, Y_dp_std, DEVICE)
        dp_r2s[split] = r2
        dp_rmses[split] = rmse
        dp_preds[split] = pred6
        dp_trues[split] = true6

    print(f"\n{'Component':<12}{'R²_train':>10}{'R²_val':>10}{'R²_test':>10}{'RMSE_test':>12}")
    print("-" * 56)
    for i, label in enumerate(DP_OUTPUT_LABELS_6):
        note = " *" if label == "Dp_zz" else "  "
        print(f"  {label:<8}{note}  {dp_r2s['train'][i]:>8.4f}  {dp_r2s['val'][i]:>8.4f}  {dp_r2s['test'][i]:>8.4f}  {dp_rmses['test'][i]:>10.2f}")
    print("-" * 56)
    print(f"  {'MEAN':<10}  {dp_r2s['train'].mean():>8.4f}  {dp_r2s['val'].mean():>8.4f}  {dp_r2s['test'].mean():>8.4f}  {dp_rmses['test'].mean():>10.2f}")


    # One-step evaluation: Rho (rate target)

    print("\nEvaluating improved rho model on all splits ...")
    rho_metrics = {}
    for split, X, Y_raw in [
        ("train", X_train_rho, Y_train_rho_raw),
        ("val",   X_val_rho,   Y_val_rho_raw),
        ("test",  X_test_rho,  Y_test_rho_raw),
    ]:
        r2, rmse, pred, true = evaluate_rho_model(rho_model, X, Y_raw, Y_rho_std, DEVICE, seq_path)
        rho_metrics[split] = dict(r2=r2, rmse=rmse)

    print(f"\n{'Split':<10}{'R²':>12}{'RMSE':>16}")
    print("-" * 38)
    for split in ["train", "val", "test"]:
        print(f"{split:<10}{rho_metrics[split]['r2']:>12.4f}{rho_metrics[split]['rmse']:>16.4e}")


    # Plots

    print("\nGenerating plots ...")
    plot_training_curves(dp_history, os.path.join(NN_OUT_DIR, "dp"))
    plot_training_curves(rho_history, os.path.join(NN_OUT_DIR, "rho"))
    plot_branch_training_detail(dp_history, os.path.join(NN_OUT_DIR, "dp_rollout_training_detail.png"),
                                "DpNet step vs rollout losses")
    plot_branch_training_detail(rho_history, os.path.join(NN_OUT_DIR, "rho_rollout_training_detail.png"),
                                "RhoNet step vs rollout losses")
    plot_dp_per_component(dp_r2s, dp_rmses, NN_OUT_DIR)


    # Rollouts

    rollout_summary_A = rollout_all_test_runs_modeA(
        dp_model=dp_model,
        seq_csv_path=seq_path,
        EP_MAX=EP_MAX,
        Y_dp_std=Y_dp_std,
        out_dir=NN_OUT_DIR,
    )

    rollout_summary_B = rollout_all_test_runs_modeB(
        dp_model=dp_model,
        rho_model=rho_model,
        seq_csv_path=seq_path,
        EP_MAX=EP_MAX,
        Y_dp_std=Y_dp_std,
        Y_rho_std=Y_rho_std,
        dt_log_mean=DT_LOG_MEAN,
        dt_log_std=DT_LOG_STD,
        out_dir=NN_OUT_DIR,
    )

    print("\nDisplaying rollout Mode A: stress vs plastic strain ...")
    plot_rollout_stress_vs_plastic_strain("rollout_modeA", NN_OUT_DIR)

    print("\nDisplaying rollout Mode B: stress vs plastic strain ...")
    plot_rollout_stress_vs_plastic_strain("rollout_modeB", NN_OUT_DIR)

    print("\nDisplaying rollout Mode B: density vs plastic strain ...")
    plot_rollout_density_vs_plastic_strain("rollout_modeB", NN_OUT_DIR)

    print(f"\nAll outputs in: {NN_OUT_DIR}")
    print("  dpnet_v7_best.pt")
    print("  rhonet_v7_best.pt")
    print("  dp_train_curves.png | rho_train_curves.png")
    print("  dp_rollout_training_detail.png | rho_rollout_training_detail.png")
    print("  dp_per_component_metrics.png")
    print("  rollout_summary_modeA.csv")
    print("  rollout_summary_modeB.csv")
    print("  rollout_modeA_<runname>.csv")
    print("  rollout_modeB_<runname>.csv")

    return (
        dp_model, rho_model,
        dp_history, rho_history,
        dp_r2s, rho_metrics,
        rollout_summary_A, rollout_summary_B
    )


if __name__ == "__main__":
    set_all_seeds(RANDOM_SEED)

    dataset_paths = build_dataset_v7()

    (
        dp_model, rho_model,
        dp_history, rho_history,
        dp_r2s, rho_metrics,
        rollout_summary_A, rollout_summary_B
    ) = train_and_evaluate_v7(
        npz_path=dataset_paths["npz_path"],
        stats_path=dataset_paths["stats_path"],
        seq_path=dataset_paths["seq_path"],
    )

In [ ]:
# LSTM CODE
import os
import re
import json
import time
import zipfile
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset


# GLOBAL CONFIG

# DDD database. Built from data/raw/ on first use so the notebook runs
# from a fresh clone of the repository.
from pathlib import Path as _Path
import zipfile as _zipfile

REPO_ROOT = _Path.cwd()
if not (REPO_ROOT / "data" / "raw").is_dir():
    REPO_ROOT = REPO_ROOT.parent

RAW_DIR  = REPO_ROOT / "data" / "raw"
ZIP_PATH = str(REPO_ROOT / "data" / "SimulationCsvs.zip")

if not _Path(ZIP_PATH).exists():
    _csvs = sorted(RAW_DIR.glob("*.csv"))
    if not _csvs:
        raise FileNotFoundError(f"No CSV files in {RAW_DIR}")
    with _zipfile.ZipFile(ZIP_PATH, "w", _zipfile.ZIP_DEFLATED) as _zf:
        for _c in _csvs:
            _zf.write(_c, arcname=_c.name)
    print(f"Built {ZIP_PATH} from {len(_csvs)} runs")

DATASET_DIR = "data/processed/lstm_v1"
NN_OUT_DIR  = "results/lstm_v1"
os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(NN_OUT_DIR, exist_ok=True)

RANDOM_SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

N_RESAMPLE   = 1000
N_VAL_RUNS   = 8
N_TEST_RUNS  = 8

WINDOW_LEN   = 24
WINDOW_STRIDE = 12
MAX_WINDOWS_PER_RUN = None

STRESS_SCALE   = 55.0e6
LOGRHO_SHIFT   = 12.0
SDOT_LOG_SHIFT = 13.76

BATCH_SIZE      = 64
MAX_EPOCHS      = 800
LR_INIT         = 2e-4
WEIGHT_DECAY    = 1e-4
GRAD_CLIP       = 5.0
ES_PATIENCE     = 120
HUBER_DELTA     = 1.0

# Dp loss weights
W_STEP_DP = 1.0
W_ROLL_DP = 0.50

# Rho loss weights
W_STEP_RHO = 1.0
W_ROLL_RHO = 0.60

DP_INPUT_DIM  = 10
RHO_INPUT_DIM = 11

DP_HIDDEN_SIZE = 128
DP_NUM_LAYERS  = 2
DP_DROPOUT     = 0.10
DP_HEAD_HIDDEN = 128

RHO_HIDDEN_SIZE = 96
RHO_NUM_LAYERS  = 2
RHO_DROPOUT     = 0.10
RHO_HEAD_HIDDEN = 96

LP_COLS  = ["Lpxx","Lpxy","Lpxz","Lpyx","Lpyy","Lpyz","Lpzx","Lpzy","Lpzz"]
SIG_COLS = ["s_xx(Pa)","s_yy(Pa)","s_zz(Pa)","s_yz(Pa)","s_zx(Pa)","s_xy(Pa)"]
EP_COL   = "ep_eq(-)"
RHO_COL  = "density(1/m^2)"

INPUT_LABELS_DP = ['σ_xx','σ_yy','σ_zz','σ_yz','σ_zx','σ_xy',
                   'σ_dot','log10ρ','loading','εp_cum']

INPUT_LABELS_RHO = ['σ_xx','σ_yy','σ_zz','σ_yz','σ_zx','σ_xy',
                    'σ_dot','log10ρ','loading','εp_cum','dt']

DP_OUTPUT_LABELS_5 = ['Dp_xx','Dp_yy','Dp_yz','Dp_xz','Dp_xy']
DP_OUTPUT_LABELS_6 = ['Dp_xx','Dp_yy','Dp_zz','Dp_yz','Dp_xz','Dp_xy']



# HELPER FUNCTIONS
def set_all_seeds(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def dp6_to_epdot_eq_np(Dp6):
    T = np.array([
        [Dp6[0], Dp6[5], Dp6[4]],
        [Dp6[5], Dp6[1], Dp6[3]],
        [Dp6[4], Dp6[3], Dp6[2]],
    ], dtype=np.float64)
    return np.sqrt((2.0 / 3.0) * np.sum(T * T))


def dp5_to_epdot_eq_torch(dp5_phys):
    dxx = dp5_phys[..., 0]
    dyy = dp5_phys[..., 1]
    dzz = -(dxx + dyy)
    dyz = dp5_phys[..., 2]
    dxz = dp5_phys[..., 3]
    dxy = dp5_phys[..., 4]
    TT = dxx*dxx + dyy*dyy + dzz*dzz + 2.0*(dxy*dxy + dxz*dxz + dyz*dyz)
    return torch.sqrt((2.0 / 3.0) * torch.clamp(TT, min=1e-40))


def reconstruct_Y6_from_Y5(Y5_phys):
    Dp_xx = Y5_phys[:, 0:1]
    Dp_yy = Y5_phys[:, 1:2]
    Dp_zz = -(Dp_xx + Dp_yy)
    Dp_yz = Y5_phys[:, 2:3]
    Dp_xz = Y5_phys[:, 3:4]
    Dp_xy = Y5_phys[:, 4:5]
    return np.hstack([Dp_xx, Dp_yy, Dp_zz, Dp_yz, Dp_xz, Dp_xy])



# DATASET BUILDING

def parse_filename(name: str):
    stem = Path(name).stem
    loading = 0 if re.search(r'(?i)uniaxial', stem) else \
              1 if re.search(r'(?i)shear', stem) else -1

    nums = [float(x) for x in re.findall(
        r'[\d]+(?:\.[\d]+)?(?:e[+-]?[\d]+)?', stem, re.IGNORECASE
    )]

    target_mpa = sigma_dot = rho0 = None
    for n in nums:
        if 10 <= n <= 200 and target_mpa is None:
            target_mpa = n * 1e6
        elif 1e12 <= n <= 1e14 and sigma_dot is None:
            sigma_dot = n
        elif 1e10 <= n <= 1e13 and rho0 is None:
            rho0 = n

    return dict(
        stem=stem,
        loading=loading,
        target_Pa=target_mpa,
        sigma_dot=sigma_dot,
        rho0=rho0,
    )


def resample_run(df, meta, n_points):
    t = df["time(s)"].values.astype(float)
    required = SIG_COLS + LP_COLS + [EP_COL, RHO_COL]

    if len(np.unique(t)) < 2:
        return None
    if not all(c in df.columns for c in required):
        return None
    if meta["sigma_dot"] is None or meta["rho0"] is None:
        return None

    t_new = np.linspace(t[0], t[-1], n_points)
    dt_uniform = t_new[1] - t_new[0] if len(t_new) > 1 else 0.0

    def interp(cols):
        arr = df[cols].values.astype(float)
        out = np.zeros((n_points, len(cols)), float)
        for j in range(arr.shape[1]):
            f = interp1d(
                t, arr[:, j], kind="linear",
                bounds_error=False,
                fill_value=(arr[0, j], arr[-1, j]),
            )
            out[:, j] = f(t_new)
        return out

    sig = interp(SIG_COLS)
    ep_cum = interp([EP_COL])

    rho = interp([RHO_COL]).reshape(-1)
    rho = np.maximum(rho, 1e6)
    log10rho = np.log10(rho)

    dlog10rho = np.zeros(n_points, dtype=float)
    dlog10rho[:-1] = log10rho[1:] - log10rho[:-1]
    dlog10rho[-1] = dlog10rho[-2] if n_points > 1 else 0.0
    dlog10rho_rate = dlog10rho / dt_uniform if dt_uniform > 0 else np.zeros_like(dlog10rho)

    Lp_raw = interp(LP_COLS)
    Lp_mat = Lp_raw.reshape(-1, 3, 3)
    Dp_mat = 0.5 * (Lp_mat + Lp_mat.transpose(0, 2, 1))
    Dp = np.stack([
        Dp_mat[:, 0, 0],
        Dp_mat[:, 1, 1],
        Dp_mat[:, 2, 2],
        Dp_mat[:, 1, 2],
        Dp_mat[:, 0, 2],
        Dp_mat[:, 0, 1],
    ], axis=1)

    return dict(
        sig=sig,
        ep_cum=ep_cum,
        rho=rho,
        log10rho=log10rho,
        dlog10rho=dlog10rho,
        dlog10rho_rate=dlog10rho_rate,
        Dp=Dp,
        sigma_dot=np.full(n_points, meta["sigma_dot"], float),
        rho0=np.full(n_points, meta["rho0"], float),
        loading=np.full(n_points, float(meta["loading"]), float),
        time=t_new,
        dt=np.full(n_points, dt_uniform, float),
        stem=meta["stem"],
        n_orig=len(df),
        strat_key=f"load{meta['loading']}_rho{meta['rho0']:.2e}",
    )


def exact_grouped_stratified_split(run_data, n_val=8, n_test=8, seed=42):
    rng = np.random.default_rng(seed)

    strata = defaultdict(list)
    for i, rd in enumerate(run_data):
        strata[rd["strat_key"]].append(i)

    print(f"\nStrata found ({len(strata)}):")
    for k, v in sorted(strata.items()):
        print(f"  {k}: {len(v)} runs")

    shuffled = {}
    for k, idxs in strata.items():
        arr = np.array(idxs)
        rng.shuffle(arr)
        shuffled[k] = arr.tolist()

    keys = sorted(strata.keys())
    val_counts = {k: 1 for k in keys}
    test_counts = {k: 1 for k in keys}

    extra_val = n_val - len(keys)
    extra_test = n_test - len(keys)
    if extra_val < 0 or extra_test < 0:
        raise ValueError("n_val and n_test must be at least number of strata.")

    def assign_extras(counts_a, counts_b, n_extra):
        if n_extra <= 0:
            return
        for _ in range(n_extra):
            best_k = None
            best_left = -1
            for k in keys:
                total = len(shuffled[k])
                used = counts_a[k] + counts_b[k]
                left = total - used
                if left > best_left:
                    best_left = left
                    best_k = k
            if best_k is None or best_left <= 0:
                raise RuntimeError("Not enough runs to allocate exact split.")
            counts_a[best_k] += 1

    assign_extras(val_counts, test_counts, extra_val)
    assign_extras(test_counts, val_counts, extra_test)

    idx_train, idx_val, idx_test = [], [], []
    for k in keys:
        arr = shuffled[k]
        nv = val_counts[k]
        nt = test_counts[k]
        n = len(arr)
        if nv + nt >= n:
            raise RuntimeError(f"Stratum {k} too small for requested exact split.")
        idx_val.extend(arr[:nv])
        idx_test.extend(arr[nv:nv+nt])
        idx_train.extend(arr[nv+nt:])

    return idx_train, idx_val, idx_test, val_counts, test_counts


def save_run_sequences(run_data, idx_train, idx_val, idx_test, out_dir):
    split_map = {}
    for i in idx_train:
        split_map[run_data[i]["stem"]] = "train"
    for i in idx_val:
        split_map[run_data[i]["stem"]] = "val"
    for i in idx_test:
        split_map[run_data[i]["stem"]] = "test"

    rows = []
    for rd in run_data:
        stem = rd["stem"]
        split = split_map[stem]
        n = rd["sig"].shape[0]
        for k in range(n):
            rows.append({
                "stem": stem,
                "split": split,
                "k": k,
                "time": rd["time"][k],
                "dt": rd["dt"][k],
                "s_xx": rd["sig"][k, 0],
                "s_yy": rd["sig"][k, 1],
                "s_zz": rd["sig"][k, 2],
                "s_yz": rd["sig"][k, 3],
                "s_zx": rd["sig"][k, 4],
                "s_xy": rd["sig"][k, 5],
                "sigma_dot": rd["sigma_dot"][k],
                "rho0": rd["rho0"][k],
                "rho_true": rd["rho"][k],
                "log10rho_true": rd["log10rho"][k],
                "dlog10rho_true": rd["dlog10rho"][k],
                "dlog10rho_rate_true": rd["dlog10rho_rate"][k],
                "loading": rd["loading"][k],
                "ep_eq_cum": rd["ep_cum"][k, 0],
                "Dp_xx": rd["Dp"][k, 0],
                "Dp_yy": rd["Dp"][k, 1],
                "Dp_zz": rd["Dp"][k, 2],
                "Dp_yz": rd["Dp"][k, 3],
                "Dp_xz": rd["Dp"][k, 4],
                "Dp_xy": rd["Dp"][k, 5],
            })

    df_seq = pd.DataFrame(rows)
    seq_path = os.path.join(out_dir, "run_sequences_lstm_v1.csv")
    df_seq.to_csv(seq_path, index=False)
    print(f"Saved run sequences → {seq_path}")
    return seq_path


def build_dataset_lstm():
    print(f"Reading {ZIP_PATH} ...")
    run_data = []

    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        csv_names = sorted([
            n for n in zf.namelist()
            if n.lower().endswith(".csv") and not n.startswith("__")
        ])
        print(f"Found {len(csv_names)} CSV files")

        for name in csv_names:
            meta = parse_filename(name)
            try:
                with zf.open(name) as f:
                    df = pd.read_csv(f, comment="#")
                df.columns = [c.strip() for c in df.columns]

                rd = resample_run(df, meta, N_RESAMPLE)
                if rd is not None:
                    run_data.append(rd)
                    print(f"  ✓ {meta['stem'][:52]:52s} orig={rd['n_orig']:6d} strat={rd['strat_key']}")
            except Exception as e:
                print(f"  ✗ {meta['stem']}: {e}")

    print(f"\nTotal runs: {len(run_data)}")

    idx_train, idx_val, idx_test, val_counts, test_counts = exact_grouped_stratified_split(
        run_data, n_val=N_VAL_RUNS, n_test=N_TEST_RUNS, seed=RANDOM_SEED
    )

    print(f"\nExact split: train={len(idx_train)}  val={len(idx_val)}  test={len(idx_test)} runs")

    EP_MAX = 0.0
    all_train_dp = []
    all_train_rho = []
    for i in idx_train:
        rd = run_data[i]
        X_dp = np.column_stack([
            rd["sig"],
            rd["sigma_dot"][:, None],
            rd["log10rho"][:, None],
            rd["loading"][:, None],
            rd["ep_cum"],
        ])
        X_rho = np.column_stack([
            rd["sig"],
            rd["sigma_dot"][:, None],
            rd["log10rho"][:, None],
            rd["loading"][:, None],
            rd["ep_cum"],
            rd["dt"][:, None],
        ])
        Y_dp = np.column_stack([
            rd["Dp"][:, 0],
            rd["Dp"][:, 1],
            rd["Dp"][:, 3],
            rd["Dp"][:, 4],
            rd["Dp"][:, 5],
        ])
        Y_rho = rd["dlog10rho_rate"][:, None]
        all_train_dp.append(Y_dp)
        all_train_rho.append(Y_rho)
        EP_MAX = max(EP_MAX, float(rd["ep_cum"].max()))

    Y_dp_train_all = np.vstack(all_train_dp)
    Y_rho_train_all = np.vstack(all_train_rho)
    Y_dp_std = Y_dp_train_all.std(axis=0).astype(np.float64)
    Y_dp_std[Y_dp_std < 1e-40] = 1.0
    Y_rho_std = float(Y_rho_train_all.std())
    if Y_rho_std < 1e-40:
        Y_rho_std = 1.0

    train_dt_logs = []
    for i in idx_train:
        rd = run_data[i]
        train_dt_logs.append(np.log10(np.maximum(rd["dt"], 1e-40)))
    dt_log_train = np.concatenate(train_dt_logs)
    DT_LOG_MEAN = float(dt_log_train.mean())
    DT_LOG_STD = float(dt_log_train.std())
    if DT_LOG_STD < 1e-12:
        DT_LOG_STD = 1.0

    stats = {
        "Y_dp_std": Y_dp_std.tolist(),
        "Y_rho_std": Y_rho_std,
        "EP_MAX": EP_MAX,
        "DT_LOG_MEAN": DT_LOG_MEAN,
        "DT_LOG_STD": DT_LOG_STD,
        "window_len": WINDOW_LEN,
        "window_stride": WINDOW_STRIDE,
        "train_runs": len(idx_train),
        "val_runs": len(idx_val),
        "test_runs": len(idx_test),
    }

    stats_path = os.path.join(DATASET_DIR, "norm_stats_lstm_v1.json")
    split_path = os.path.join(DATASET_DIR, "run_split_lstm_v1.csv")
    seq_path = save_run_sequences(run_data, idx_train, idx_val, idx_test, DATASET_DIR)

    with open(stats_path, "w") as f:
        json.dump(stats, f, indent=2)

    rows = []
    for label, idxs in [("train", idx_train), ("val", idx_val), ("test", idx_test)]:
        for i in idxs:
            rows.append({
                "stem": run_data[i]["stem"],
                "split": label,
                "strat_key": run_data[i]["strat_key"],
            })
    pd.DataFrame(rows).to_csv(split_path, index=False)

    print("\nSaved:")
    print(f"  {stats_path}")
    print(f"  {split_path}")
    print(f"  {seq_path}")

    return {
        "stats_path": stats_path,
        "split_path": split_path,
        "seq_path": seq_path,
    }



# NORMALIZATION

def normalize_X_dp(X_raw, EP_MAX):
    X = X_raw.copy().astype(np.float64)
    X[..., :6] = X[..., :6] / STRESS_SCALE
    X[..., 6] = np.log10(np.abs(X[..., 6]) + 1e-40) - SDOT_LOG_SHIFT
    X[..., 7] = X[..., 7] - LOGRHO_SHIFT
    X[..., 8] = 2.0 * X[..., 8] - 1.0
    X[..., 9] = X[..., 9] / (EP_MAX + 1e-40)
    return X.astype(np.float32)


def normalize_X_rho(X_raw, EP_MAX, dt_log_mean, dt_log_std):
    X = X_raw.copy().astype(np.float64)
    X[..., :6] = X[..., :6] / STRESS_SCALE
    X[..., 6] = np.log10(np.abs(X[..., 6]) + 1e-40) - SDOT_LOG_SHIFT
    X[..., 7] = X[..., 7] - LOGRHO_SHIFT
    X[..., 8] = 2.0 * X[..., 8] - 1.0
    X[..., 9] = X[..., 9] / (EP_MAX + 1e-40)
    dt_log = np.log10(np.maximum(X[..., 10], 1e-40))
    X[..., 10] = (dt_log - dt_log_mean) / (dt_log_std + 1e-40)
    return X.astype(np.float32)


def normalize_Y_dp(Y_raw, Y_std):
    return (Y_raw / (Y_std + 1e-40)).astype(np.float32)


def denormalize_Y_dp(Y_norm, Y_std):
    return Y_norm * Y_std


def normalize_Y_rho(Y_raw, Y_std):
    return (Y_raw / (Y_std + 1e-40)).astype(np.float32)


def denormalize_Y_rho(Y_norm, Y_std):
    return Y_norm * Y_std


# SEQUENCE DATASETS

class DpSequenceDataset(Dataset):
    def __init__(self, seq_csv_path, stats, split="train", window_len=24, stride=12,
                 max_windows_per_run=None):
        self.window_len = int(window_len)
        self.stride = int(stride)
        self.stats = stats
        self.windows = []

        df = pd.read_csv(seq_csv_path)
        df = df[df["split"] == split].copy()

        for stem, g in df.groupby("stem"):
            g = g.sort_values("k").reset_index(drop=True)
            n = len(g)
            count_for_run = 0
            for start in range(0, n - self.window_len + 1, self.stride):
                end = start + self.window_len
                sub = g.iloc[start:end]

                x_raw = np.column_stack([
                    sub["s_xx"].values,
                    sub["s_yy"].values,
                    sub["s_zz"].values,
                    sub["s_yz"].values,
                    sub["s_zx"].values,
                    sub["s_xy"].values,
                    sub["sigma_dot"].values,
                    sub["log10rho_true"].values,
                    sub["loading"].values,
                    sub["ep_eq_cum"].values,
                ]).astype(np.float32)

                y_raw = np.column_stack([
                    sub["Dp_xx"].values,
                    sub["Dp_yy"].values,
                    sub["Dp_yz"].values,
                    sub["Dp_xz"].values,
                    sub["Dp_xy"].values,
                ]).astype(np.float32)

                x_norm = normalize_X_dp(x_raw, stats["EP_MAX"])
                y_norm = normalize_Y_dp(y_raw, np.array(stats["Y_dp_std"], dtype=np.float64))

                self.windows.append({
                    "stem": stem,
                    "x_raw": x_raw,
                    "x_norm": x_norm,
                    "y_raw": y_raw,
                    "y_norm": y_norm,
                    "dt": sub["dt"].values.astype(np.float32),
                    "ep_true": sub["ep_eq_cum"].values.astype(np.float32),
                    "log10rho_true": sub["log10rho_true"].values.astype(np.float32),
                })

                count_for_run += 1
                if max_windows_per_run is not None and count_for_run >= max_windows_per_run:
                    break

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        item = self.windows[idx]
        return (
            torch.tensor(item["x_raw"], dtype=torch.float32),
            torch.tensor(item["x_norm"], dtype=torch.float32),
            torch.tensor(item["y_raw"], dtype=torch.float32),
            torch.tensor(item["y_norm"], dtype=torch.float32),
            torch.tensor(item["dt"], dtype=torch.float32),
            torch.tensor(item["ep_true"], dtype=torch.float32),
            torch.tensor(item["log10rho_true"], dtype=torch.float32),
        )


class RhoSequenceDataset(Dataset):
    def __init__(self, seq_csv_path, stats, split="train", window_len=24, stride=12,
                 max_windows_per_run=None):
        self.window_len = int(window_len)
        self.stride = int(stride)
        self.stats = stats
        self.windows = []

        df = pd.read_csv(seq_csv_path)
        df = df[df["split"] == split].copy()

        for stem, g in df.groupby("stem"):
            g = g.sort_values("k").reset_index(drop=True)
            n = len(g)
            count_for_run = 0
            for start in range(0, n - self.window_len + 1, self.stride):
                end = start + self.window_len
                sub = g.iloc[start:end]

                x_raw = np.column_stack([
                    sub["s_xx"].values,
                    sub["s_yy"].values,
                    sub["s_zz"].values,
                    sub["s_yz"].values,
                    sub["s_zx"].values,
                    sub["s_xy"].values,
                    sub["sigma_dot"].values,
                    sub["log10rho_true"].values,
                    sub["loading"].values,
                    sub["ep_eq_cum"].values,
                    sub["dt"].values,
                ]).astype(np.float32)

                y_raw = sub["dlog10rho_rate_true"].values.astype(np.float32)[:, None]

                x_norm = normalize_X_rho(
                    x_raw,
                    stats["EP_MAX"],
                    stats["DT_LOG_MEAN"],
                    stats["DT_LOG_STD"],
                )
                y_norm = normalize_Y_rho(y_raw, stats["Y_rho_std"])

                self.windows.append({
                    "stem": stem,
                    "x_raw": x_raw,
                    "x_norm": x_norm,
                    "y_raw": y_raw,
                    "y_norm": y_norm,
                    "dt": sub["dt"].values.astype(np.float32),
                    "ep_true": sub["ep_eq_cum"].values.astype(np.float32),
                    "log10rho_true": sub["log10rho_true"].values.astype(np.float32),
                })

                count_for_run += 1
                if max_windows_per_run is not None and count_for_run >= max_windows_per_run:
                    break

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        item = self.windows[idx]
        return (
            torch.tensor(item["x_raw"], dtype=torch.float32),
            torch.tensor(item["x_norm"], dtype=torch.float32),
            torch.tensor(item["y_raw"], dtype=torch.float32),
            torch.tensor(item["y_norm"], dtype=torch.float32),
            torch.tensor(item["dt"], dtype=torch.float32),
            torch.tensor(item["ep_true"], dtype=torch.float32),
            torch.tensor(item["log10rho_true"], dtype=torch.float32),
        )



# MODELS

class LSTMRegressor(nn.Module):
    def __init__(self, input_dim, hidden_size, num_layers, output_dim,
                 head_hidden=128, dropout=0.1):
        super().__init__()
        lstm_dropout = dropout if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=lstm_dropout,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_size, head_hidden),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden, output_dim),
        )
        self._init_weights()

    def _init_weights(self):
        for name, param in self.named_parameters():
            if "weight" in name:
                if param.ndim >= 2:
                    nn.init.xavier_uniform_(param)
                else:
                    nn.init.uniform_(param, -0.05, 0.05)
            elif "bias" in name:
                nn.init.zeros_(param)

    def forward(self, x_seq):
        out, _ = self.lstm(x_seq)
        y = self.head(out)
        return y


# LOSSES

class WeightedHuberLoss(nn.Module):
    def __init__(self, Y_std, delta=1.0):
        super().__init__()
        Y_std = np.asarray(Y_std, dtype=np.float32).reshape(-1)
        w = 1.0 / (Y_std + 1e-40)
        w = w / w.sum() * len(Y_std)
        self.register_buffer("weights", torch.tensor(w, dtype=torch.float32))
        self.huber = nn.HuberLoss(reduction="none", delta=delta)

    def forward(self, pred, target):
        return (self.huber(pred, target) * self.weights).mean()



# Dp TRAINING / EVAL

def rollout_window_losses_dp_lstm(model, x_raw, dt, ep_true, y_true_raw, stats, device):
    B, T, _ = x_raw.shape
    x_raw = x_raw.to(device)
    dt = dt.to(device)
    ep_true = ep_true.to(device)
    y_true_raw = y_true_raw.to(device)
    y_dp_std_t = torch.tensor(stats["Y_dp_std"], dtype=torch.float32, device=device)

    x_norm_np = normalize_X_dp(x_raw.detach().cpu().numpy(), stats["EP_MAX"])
    x_norm = torch.tensor(x_norm_np, dtype=torch.float32, device=device)
    pred_norm = model(x_norm)
    pred_phys = pred_norm * y_dp_std_t.view(1, 1, -1)
    step_loss = nn.HuberLoss(reduction="mean", delta=HUBER_DELTA)(pred_phys, y_true_raw)

    ep_pred = torch.zeros((B, T), dtype=torch.float32, device=device)
    ep_pred[:, 0] = ep_true[:, 0]

    for t in range(T):
        x_roll = x_raw[:, :t+1, :].clone()
        x_roll[:, t, 9] = ep_pred[:, t]
        x_roll_np = normalize_X_dp(x_roll.detach().cpu().numpy(), stats["EP_MAX"])
        x_roll_norm = torch.tensor(x_roll_np, dtype=torch.float32, device=device)
        pred_seq_norm = model(x_roll_norm)
        pred_t_phys = pred_seq_norm[:, -1, :] * y_dp_std_t.view(1, -1)
        epdot_t = dp5_to_epdot_eq_torch(pred_t_phys)
        if t + 1 < T:
            ep_pred[:, t + 1] = ep_pred[:, t] + epdot_t * dt[:, t]

    roll_loss = nn.HuberLoss(reduction="mean", delta=HUBER_DELTA)(ep_pred, ep_true)
    return step_loss, roll_loss


def run_epoch_train_dp_lstm(model, loader, optimizer, stats, device):
    model.train()
    step_loss_fn = WeightedHuberLoss(stats["Y_dp_std"], delta=HUBER_DELTA).to(device)
    total, total_step, total_roll, n = 0.0, 0.0, 0.0, 0

    for x_raw, x_norm, y_raw, y_norm, dt, ep_true, _ in loader:
        x_norm = x_norm.to(device)
        y_norm = y_norm.to(device)
        pred_norm = model(x_norm)
        loss_step_main = step_loss_fn(pred_norm, y_norm)

        loss_step_roll, loss_roll = rollout_window_losses_dp_lstm(
            model, x_raw, dt, ep_true, y_raw, stats, device
        )

        combined = W_STEP_DP * loss_step_main + 0.25 * (W_STEP_DP * loss_step_roll + W_ROLL_DP * loss_roll)
        optimizer.zero_grad()
        combined.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()

        total += combined.item()
        total_step += loss_step_main.item()
        total_roll += loss_roll.item()
        n += 1

    return total / max(n, 1), total_step / max(n, 1), total_roll / max(n, 1)


def run_epoch_eval_dp_lstm(model, loader, stats, device):
    model.eval()
    step_loss_fn = WeightedHuberLoss(stats["Y_dp_std"], delta=HUBER_DELTA).to(device)
    total, total_step, total_roll, n = 0.0, 0.0, 0.0, 0

    with torch.no_grad():
        for x_raw, x_norm, y_raw, y_norm, dt, ep_true, _ in loader:
            x_norm = x_norm.to(device)
            y_norm = y_norm.to(device)
            pred_norm = model(x_norm)
            loss_step_main = step_loss_fn(pred_norm, y_norm)

            loss_step_roll, loss_roll = rollout_window_losses_dp_lstm(
                model, x_raw, dt, ep_true, y_raw, stats, device
            )

            combined = W_STEP_DP * loss_step_main + 0.25 * (W_STEP_DP * loss_step_roll + W_ROLL_DP * loss_roll)
            total += combined.item()
            total_step += loss_step_main.item()
            total_roll += loss_roll.item()
            n += 1

    return total / max(n, 1), total_step / max(n, 1), total_roll / max(n, 1)



# RHO TRAINING / EVAL

def rollout_window_losses_rho_lstm(model, x_raw, dt, log10rho_true, y_true_raw, stats, device):
    B, T, _ = x_raw.shape
    x_raw = x_raw.to(device)
    dt = dt.to(device)
    log10rho_true = log10rho_true.to(device)
    y_true_raw = y_true_raw.to(device)
    y_rho_std_t = torch.tensor(stats["Y_rho_std"], dtype=torch.float32, device=device)

    x_norm_np = normalize_X_rho(
        x_raw.detach().cpu().numpy(),
        stats["EP_MAX"], stats["DT_LOG_MEAN"], stats["DT_LOG_STD"]
    )
    x_norm = torch.tensor(x_norm_np, dtype=torch.float32, device=device)
    pred_norm = model(x_norm)
    pred_phys = pred_norm * y_rho_std_t
    step_loss = nn.HuberLoss(reduction="mean", delta=HUBER_DELTA)(pred_phys, y_true_raw)

    log10rho_pred = torch.zeros((B, T), dtype=torch.float32, device=device)
    log10rho_pred[:, 0] = log10rho_true[:, 0]

    for t in range(T):
        x_roll = x_raw[:, :t+1, :].clone()
        x_roll[:, t, 7] = log10rho_pred[:, t]
        x_roll_np = normalize_X_rho(
            x_roll.detach().cpu().numpy(),
            stats["EP_MAX"], stats["DT_LOG_MEAN"], stats["DT_LOG_STD"]
        )
        x_roll_norm = torch.tensor(x_roll_np, dtype=torch.float32, device=device)
        pred_seq_norm = model(x_roll_norm)
        pred_t_phys = pred_seq_norm[:, -1, 0] * y_rho_std_t
        if t + 1 < T:
            log10rho_pred[:, t + 1] = log10rho_pred[:, t] + pred_t_phys * dt[:, t]

    roll_loss = nn.HuberLoss(reduction="mean", delta=HUBER_DELTA)(log10rho_pred, log10rho_true)
    return step_loss, roll_loss


def run_epoch_train_rho_lstm(model, loader, optimizer, stats, device):
    model.train()
    step_loss_fn = nn.HuberLoss(reduction="mean", delta=HUBER_DELTA)
    total, total_step, total_roll, n = 0.0, 0.0, 0.0, 0

    for x_raw, x_norm, y_raw, y_norm, dt, _, log10rho_true in loader:
        x_norm = x_norm.to(device)
        y_norm = y_norm.to(device)
        pred_norm = model(x_norm)
        loss_step_main = step_loss_fn(pred_norm, y_norm)

        loss_step_roll, loss_roll = rollout_window_losses_rho_lstm(
            model, x_raw, dt, log10rho_true, y_raw, stats, device
        )

        combined = W_STEP_RHO * loss_step_main + 0.30 * (W_STEP_RHO * loss_step_roll + W_ROLL_RHO * loss_roll)
        optimizer.zero_grad()
        combined.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()

        total += combined.item()
        total_step += loss_step_main.item()
        total_roll += loss_roll.item()
        n += 1

    return total / max(n, 1), total_step / max(n, 1), total_roll / max(n, 1)


def run_epoch_eval_rho_lstm(model, loader, stats, device):
    model.eval()
    step_loss_fn = nn.HuberLoss(reduction="mean", delta=HUBER_DELTA)
    total, total_step, total_roll, n = 0.0, 0.0, 0.0, 0

    with torch.no_grad():
        for x_raw, x_norm, y_raw, y_norm, dt, _, log10rho_true in loader:
            x_norm = x_norm.to(device)
            y_norm = y_norm.to(device)
            pred_norm = model(x_norm)
            loss_step_main = step_loss_fn(pred_norm, y_norm)

            loss_step_roll, loss_roll = rollout_window_losses_rho_lstm(
                model, x_raw, dt, log10rho_true, y_raw, stats, device
            )

            combined = W_STEP_RHO * loss_step_main + 0.30 * (W_STEP_RHO * loss_step_roll + W_ROLL_RHO * loss_roll)
            total += combined.item()
            total_step += loss_step_main.item()
            total_roll += loss_roll.item()
            n += 1

    return total / max(n, 1), total_step / max(n, 1), total_roll / max(n, 1)



# TRAIN WRAPPERS

def train_dp_lstm(model, train_loader, val_loader, stats, out_path):
    optimizer = torch.optim.Adam(model.parameters(), lr=LR_INIT, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=20, min_lr=1e-6
    )

    history = dict(train=[], val=[], train_step=[], val_step=[], train_roll=[], val_roll=[], lr=[], best_epoch=0)
    best_val = float("inf")
    best_state = None
    es_counter = 0
    t0_wall = time.time()

    print(f"\nTraining Dp-LSTM up to {MAX_EPOCHS} epochs ...")
    for epoch in range(1, MAX_EPOCHS + 1):
        tr, tr_step, tr_roll = run_epoch_train_dp_lstm(model, train_loader, optimizer, stats, DEVICE)
        va, va_step, va_roll = run_epoch_eval_dp_lstm(model, val_loader, stats, DEVICE)
        scheduler.step(va)
        lr = optimizer.param_groups[0]["lr"]

        history["train"].append(tr)
        history["val"].append(va)
        history["train_step"].append(tr_step)
        history["val_step"].append(va_step)
        history["train_roll"].append(tr_roll)
        history["val_roll"].append(va_roll)
        history["lr"].append(lr)

        if va < best_val:
            best_val = va
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            history["best_epoch"] = epoch
            es_counter = 0
        else:
            es_counter += 1

        if epoch % 25 == 0 or epoch == 1:
            print(
                f"  Epoch {epoch:4d} | train={tr:.5f} val={va:.5f} | "
                f"step(tr/va)=({tr_step:.5f}/{va_step:.5f}) | "
                f"roll(tr/va)=({tr_roll:.5f}/{va_roll:.5f}) | "
                f"best={best_val:.5f} | LR={lr:.2e} | ES={es_counter}/{ES_PATIENCE} | {time.time()-t0_wall:.0f}s"
            )

        if es_counter >= ES_PATIENCE:
            print(f"\nEarly stop Dp-LSTM at epoch {epoch} (best={best_val:.5f} @ {history['best_epoch']})")
            break

    model.load_state_dict(best_state)
    torch.save(best_state, out_path)
    print(f"Saved Dp-LSTM weights → {out_path}")
    return model, history, best_val


def train_rho_lstm(model, train_loader, val_loader, stats, out_path):
    optimizer = torch.optim.Adam(model.parameters(), lr=LR_INIT, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=20, min_lr=1e-6
    )

    history = dict(train=[], val=[], train_step=[], val_step=[], train_roll=[], val_roll=[], lr=[], best_epoch=0)
    best_val = float("inf")
    best_state = None
    es_counter = 0
    t0_wall = time.time()

    print(f"\nTraining Rho-LSTM up to {MAX_EPOCHS} epochs ...")
    for epoch in range(1, MAX_EPOCHS + 1):
        tr, tr_step, tr_roll = run_epoch_train_rho_lstm(model, train_loader, optimizer, stats, DEVICE)
        va, va_step, va_roll = run_epoch_eval_rho_lstm(model, val_loader, stats, DEVICE)
        scheduler.step(va)
        lr = optimizer.param_groups[0]["lr"]

        history["train"].append(tr)
        history["val"].append(va)
        history["train_step"].append(tr_step)
        history["val_step"].append(va_step)
        history["train_roll"].append(tr_roll)
        history["val_roll"].append(va_roll)
        history["lr"].append(lr)

        if va < best_val:
            best_val = va
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            history["best_epoch"] = epoch
            es_counter = 0
        else:
            es_counter += 1

        if epoch % 25 == 0 or epoch == 1:
            print(
                f"  Epoch {epoch:4d} | train={tr:.5f} val={va:.5f} | "
                f"step(tr/va)=({tr_step:.5f}/{va_step:.5f}) | "
                f"roll(tr/va)=({tr_roll:.5f}/{va_roll:.5f}) | "
                f"best={best_val:.5f} | LR={lr:.2e} | ES={es_counter}/{ES_PATIENCE} | {time.time()-t0_wall:.0f}s"
            )

        if es_counter >= ES_PATIENCE:
            print(f"\nEarly stop Rho-LSTM at epoch {epoch} (best={best_val:.5f} @ {history['best_epoch']})")
            break

    model.load_state_dict(best_state)
    torch.save(best_state, out_path)
    print(f"Saved Rho-LSTM weights → {out_path}")
    return model, history, best_val


# ============================================================
# METRICS
# ============================================================
def evaluate_dp_lstm(model, loader, stats, device):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for _, x_norm, y_raw, _, _, _, _ in loader:
            x_norm = x_norm.to(device)
            pred_norm = model(x_norm).cpu().numpy().reshape(-1, 5)
            pred_phys = denormalize_Y_dp(pred_norm, np.array(stats["Y_dp_std"], dtype=np.float64))
            true_phys = y_raw.numpy().reshape(-1, 5)
            preds.append(pred_phys)
            trues.append(true_phys)

    pred5 = np.vstack(preds)
    true5 = np.vstack(trues)
    pred6 = reconstruct_Y6_from_Y5(pred5)
    true6 = np.column_stack([
        true5[:, 0],
        true5[:, 1],
        -(true5[:, 0] + true5[:, 1]),
        true5[:, 2],
        true5[:, 3],
        true5[:, 4],
    ])

    r2 = np.zeros(6)
    rmse = np.zeros(6)
    for i in range(6):
        ss_res = np.sum((true6[:, i] - pred6[:, i])**2)
        ss_tot = np.sum((true6[:, i] - true6[:, i].mean())**2)
        r2[i] = 1.0 - ss_res / (ss_tot + 1e-40)
        rmse[i] = np.sqrt(np.mean((true6[:, i] - pred6[:, i])**2))

    return r2, rmse, pred6, true6


def evaluate_rho_lstm(model, loader, stats, device):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for _, x_norm, y_raw, _, _, _, _ in loader:
            x_norm = x_norm.to(device)
            pred_norm = model(x_norm).cpu().numpy().reshape(-1, 1)
            pred_phys = denormalize_Y_rho(pred_norm, stats["Y_rho_std"])
            true_phys = y_raw.numpy().reshape(-1, 1)
            preds.append(pred_phys)
            trues.append(true_phys)

    pred = np.vstack(preds)
    true = np.vstack(trues)
    ss_res = np.sum((true[:, 0] - pred[:, 0])**2)
    ss_tot = np.sum((true[:, 0] - true[:, 0].mean())**2)
    r2 = 1.0 - ss_res / (ss_tot + 1e-40)
    rmse = np.sqrt(np.mean((true[:, 0] - pred[:, 0])**2))
    return r2, rmse, pred, true

# AUTOREGRESSIVE INFERENCE

def predict_dp_once_lstm(dp_model, seq_raw, stats):
    seq_norm = normalize_X_dp(seq_raw, stats["EP_MAX"])
    x = torch.tensor(seq_norm[None, :, :], dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        pred_norm = dp_model(x).cpu().numpy()[0, -1, :]
    pred_phys = denormalize_Y_dp(pred_norm[None, :], np.array(stats["Y_dp_std"], dtype=np.float64))
    pred6 = reconstruct_Y6_from_Y5(pred_phys)
    return pred6[0]


def predict_rho_rate_once_lstm(rho_model, seq_raw, stats):
    seq_norm = normalize_X_rho(seq_raw, stats["EP_MAX"], stats["DT_LOG_MEAN"], stats["DT_LOG_STD"])
    x = torch.tensor(seq_norm[None, :, :], dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        pred_norm = rho_model(x).cpu().numpy()[0, -1, 0]
    pred_phys = denormalize_Y_rho(np.array([[pred_norm]]), stats["Y_rho_std"])
    return float(pred_phys[0, 0])


# ROLLOUT EVALUATION ON FULL RUNS

def rollout_one_run_modeA_lstm(dp_model, run_df, stats, window_len=24):
    run_df = run_df.sort_values("k").reset_index(drop=True).copy()
    ep_true = run_df["ep_eq_cum"].values.astype(np.float64)
    log10rho_true = run_df["log10rho_true"].values.astype(np.float64)
    n = len(run_df)

    ep_pred = np.zeros(n, dtype=np.float64)
    ep_pred[0] = ep_true[0]
    rows = []

    for i in range(n):
        start = max(0, i - window_len + 1)
        sub = run_df.iloc[start:i+1].copy()

        seq_raw = np.column_stack([
            sub["s_xx"].values,
            sub["s_yy"].values,
            sub["s_zz"].values,
            sub["s_yz"].values,
            sub["s_zx"].values,
            sub["s_xy"].values,
            sub["sigma_dot"].values,
            sub["log10rho_true"].values,
            sub["loading"].values,
            sub["ep_eq_cum"].values,
        ]).astype(np.float64)
        seq_raw[-1, 9] = ep_pred[i]

        pred6 = predict_dp_once_lstm(dp_model, seq_raw, stats)
        epdot_eq = dp6_to_epdot_eq_np(pred6)

        row = run_df.iloc[i]
        rows.append({
            "stem": row["stem"],
            "k": int(row["k"]),
            "time": float(row["time"]),
            "dt": float(row["dt"]),
            "s_xx": float(row["s_xx"]),
            "s_yy": float(row["s_yy"]),
            "s_zz": float(row["s_zz"]),
            "s_yz": float(row["s_yz"]),
            "s_zx": float(row["s_zx"]),
            "s_xy": float(row["s_xy"]),
            "ep_eq_true": float(ep_true[i]),
            "ep_eq_pred": float(ep_pred[i]),
            "epdot_eq_pred": float(epdot_eq),
            "log10rho_true": float(log10rho_true[i]),
            "rho_true": float(10.0 ** log10rho_true[i]),
            "Dp_xx_pred": float(pred6[0]),
            "Dp_yy_pred": float(pred6[1]),
            "Dp_zz_pred": float(pred6[2]),
            "Dp_yz_pred": float(pred6[3]),
            "Dp_xz_pred": float(pred6[4]),
            "Dp_xy_pred": float(pred6[5]),
            "Dp_xx_true": float(row["Dp_xx"]),
            "Dp_yy_true": float(row["Dp_yy"]),
            "Dp_zz_true": float(row["Dp_zz"]),
            "Dp_yz_true": float(row["Dp_yz"]),
            "Dp_xz_true": float(row["Dp_xz"]),
            "Dp_xy_true": float(row["Dp_xy"]),
        })

        if i + 1 < n:
            ep_pred[i + 1] = ep_pred[i] + epdot_eq * float(row["dt"])

    return pd.DataFrame(rows)


def rollout_one_run_modeB_lstm(dp_model, rho_model, run_df, stats, window_len=24):
    run_df = run_df.sort_values("k").reset_index(drop=True).copy()
    ep_true = run_df["ep_eq_cum"].values.astype(np.float64)
    log10rho_true = run_df["log10rho_true"].values.astype(np.float64)
    n = len(run_df)

    ep_pred = np.zeros(n, dtype=np.float64)
    log10rho_pred = np.zeros(n, dtype=np.float64)
    ep_pred[0] = ep_true[0]
    log10rho_pred[0] = log10rho_true[0]
    rows = []

    for i in range(n):
        start = max(0, i - window_len + 1)
        sub = run_df.iloc[start:i+1].copy()

        seq_dp = np.column_stack([
            sub["s_xx"].values,
            sub["s_yy"].values,
            sub["s_zz"].values,
            sub["s_yz"].values,
            sub["s_zx"].values,
            sub["s_xy"].values,
            sub["sigma_dot"].values,
            sub["log10rho_true"].values,
            sub["loading"].values,
            sub["ep_eq_cum"].values,
        ]).astype(np.float64)
        seq_dp[-1, 7] = log10rho_pred[i]
        seq_dp[-1, 9] = ep_pred[i]
        pred6 = predict_dp_once_lstm(dp_model, seq_dp, stats)
        epdot_eq = dp6_to_epdot_eq_np(pred6)

        seq_rho = np.column_stack([
            sub["s_xx"].values,
            sub["s_yy"].values,
            sub["s_zz"].values,
            sub["s_yz"].values,
            sub["s_zx"].values,
            sub["s_xy"].values,
            sub["sigma_dot"].values,
            sub["log10rho_true"].values,
            sub["loading"].values,
            sub["ep_eq_cum"].values,
            sub["dt"].values,
        ]).astype(np.float64)
        seq_rho[-1, 7] = log10rho_pred[i]
        seq_rho[-1, 9] = ep_pred[i]
        rho_rate_pred = predict_rho_rate_once_lstm(rho_model, seq_rho, stats)

        row = run_df.iloc[i]
        rows.append({
            "stem": row["stem"],
            "k": int(row["k"]),
            "time": float(row["time"]),
            "dt": float(row["dt"]),
            "s_xx": float(row["s_xx"]),
            "s_yy": float(row["s_yy"]),
            "s_zz": float(row["s_zz"]),
            "s_yz": float(row["s_yz"]),
            "s_zx": float(row["s_zx"]),
            "s_xy": float(row["s_xy"]),
            "ep_eq_true": float(ep_true[i]),
            "ep_eq_pred": float(ep_pred[i]),
            "epdot_eq_pred": float(epdot_eq),
            "log10rho_true": float(log10rho_true[i]),
            "log10rho_pred": float(log10rho_pred[i]),
            "rho_true": float(10.0 ** log10rho_true[i]),
            "rho_pred": float(10.0 ** log10rho_pred[i]),
            "dlog10rho_true": float(row["dlog10rho_true"]),
            "dlog10rho_rate_true": float(row["dlog10rho_rate_true"]),
            "rho_rate_pred": float(rho_rate_pred),
            "Dp_xx_pred": float(pred6[0]),
            "Dp_yy_pred": float(pred6[1]),
            "Dp_zz_pred": float(pred6[2]),
            "Dp_yz_pred": float(pred6[3]),
            "Dp_xz_pred": float(pred6[4]),
            "Dp_xy_pred": float(pred6[5]),
            "Dp_xx_true": float(row["Dp_xx"]),
            "Dp_yy_true": float(row["Dp_yy"]),
            "Dp_zz_true": float(row["Dp_zz"]),
            "Dp_yz_true": float(row["Dp_yz"]),
            "Dp_xz_true": float(row["Dp_xz"]),
            "Dp_xy_true": float(row["Dp_xy"]),
        })

        if i + 1 < n:
            dt_i = float(row["dt"])
            ep_pred[i + 1] = ep_pred[i] + epdot_eq * dt_i
            log10rho_pred[i + 1] = log10rho_pred[i] + rho_rate_pred * dt_i

    return pd.DataFrame(rows)


def rollout_all_test_runs_modeA_lstm(dp_model, seq_csv_path, stats, out_dir):
    df_seq = pd.read_csv(seq_csv_path)
    test_runs = sorted(df_seq.loc[df_seq["split"] == "test", "stem"].unique())
    all_summary = []

    for stem in test_runs:
        run_df = df_seq[(df_seq["split"] == "test") & (df_seq["stem"] == stem)].copy()
        pred_df = rollout_one_run_modeA_lstm(dp_model, run_df, stats, window_len=WINDOW_LEN)
        ep_rel_err = np.mean(
            np.abs(pred_df["ep_eq_pred"] - pred_df["ep_eq_true"]) /
            np.maximum(np.abs(pred_df["ep_eq_true"]), 1e-30)
        )
        all_summary.append({"stem": stem, "ep_eq_rel_err": ep_rel_err})
        safe_stem = re.sub(r"[^A-Za-z0-9_\-]+", "_", stem)
        pred_df.to_csv(os.path.join(out_dir, f"rollout_modeA_{safe_stem}.csv"), index=False)

    summary_df = pd.DataFrame(all_summary)
    summary_df.to_csv(os.path.join(out_dir, "rollout_summary_modeA.csv"), index=False)
    print("\nMode A rollout summary:")
    print(summary_df)
    print("\nMean rollout ep_eq error:", summary_df["ep_eq_rel_err"].mean())
    return summary_df


def rollout_all_test_runs_modeB_lstm(dp_model, rho_model, seq_csv_path, stats, out_dir):
    df_seq = pd.read_csv(seq_csv_path)
    test_runs = sorted(df_seq.loc[df_seq["split"] == "test", "stem"].unique())
    all_summary = []

    for stem in test_runs:
        run_df = df_seq[(df_seq["split"] == "test") & (df_seq["stem"] == stem)].copy()
        pred_df = rollout_one_run_modeB_lstm(dp_model, rho_model, run_df, stats, window_len=WINDOW_LEN)
        ep_rel_err = np.mean(
            np.abs(pred_df["ep_eq_pred"] - pred_df["ep_eq_true"]) /
            np.maximum(np.abs(pred_df["ep_eq_true"]), 1e-30)
        )
        rho_rel_err = np.mean(
            np.abs(pred_df["rho_pred"] - pred_df["rho_true"]) /
            np.maximum(np.abs(pred_df["rho_true"]), 1e-30)
        )
        all_summary.append({"stem": stem, "ep_eq_rel_err": ep_rel_err, "rho_rel_err": rho_rel_err})
        safe_stem = re.sub(r"[^A-Za-z0-9_\-]+", "_", stem)
        pred_df.to_csv(os.path.join(out_dir, f"rollout_modeB_{safe_stem}.csv"), index=False)

    summary_df = pd.DataFrame(all_summary)
    summary_df.to_csv(os.path.join(out_dir, "rollout_summary_modeB.csv"), index=False)
    print("\nMode B rollout summary:")
    print(summary_df)
    print("\nMean rollout ep_eq error:", summary_df["ep_eq_rel_err"].mean())
    print("Mean rollout rho error:", summary_df["rho_rel_err"].mean())
    return summary_df



# PLOTTING

def plot_training_curves(history, out_path_prefix):
    epochs = range(1, len(history["train"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].plot(epochs, history["train"], label="train")
    axes[0].plot(epochs, history["val"], label="val")
    axes[0].axvline(history["best_epoch"], color="red", ls=":", label=f"best={history['best_epoch']}")
    axes[0].set_yscale("log")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Loss curves")
    axes[0].legend()

    axes[1].plot(epochs, history["lr"])
    axes[1].set_yscale("log")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("LR")
    axes[1].set_title("Learning-rate history")

    plt.tight_layout()
    plt.savefig(out_path_prefix + "_train_curves.png", dpi=150)
    plt.close()


def plot_branch_training_detail(history, out_path, title):
    epochs = range(1, len(history["train"]) + 1)
    plt.figure(figsize=(12, 4))
    plt.plot(epochs, history["train_step"], label="train step")
    plt.plot(epochs, history["val_step"], label="val step")
    plt.plot(epochs, history["train_roll"], label="train roll")
    plt.plot(epochs, history["val_roll"], label="val roll")
    plt.yscale("log")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_dp_per_component(r2s, rmses, out_dir):
    x = np.arange(6)
    w = 0.25
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for j, split in enumerate(["train", "val", "test"]):
        axes[0].bar(x + (j - 1) * w, r2s[split], w, label=split)
        axes[1].bar(x + (j - 1) * w, rmses[split], w, label=split)

    axes[0].axhline(1.0, color="k", ls="--", lw=0.8)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(DP_OUTPUT_LABELS_6)
    axes[0].set_ylabel("R²")
    axes[0].set_title("Dp-LSTM: R² per component")
    axes[0].legend()

    axes[1].set_xticks(x)
    axes[1].set_xticklabels(DP_OUTPUT_LABELS_6)
    axes[1].set_ylabel("RMSE")
    axes[1].set_title("Dp-LSTM: RMSE per component")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "dp_per_component_metrics.png"), dpi=150)
    plt.close()


def plot_rollout_stress_vs_plastic_strain(pattern_prefix, out_dir, max_plots=None):
    import glob
    csv_files = sorted(glob.glob(os.path.join(out_dir, f"{pattern_prefix}_*.csv")))
    csv_files = [f for f in csv_files if "summary" not in os.path.basename(f).lower()]
    if not csv_files:
        print(f"No rollout CSV files found for {pattern_prefix}.")
        return
    if max_plots is not None:
        csv_files = csv_files[:max_plots]

    n = len(csv_files)
    ncols = 2
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4.5 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for ax, fpath in zip(axes, csv_files):
        df = pd.read_csv(fpath)
        stem = str(df["stem"].iloc[0])
        if "uniaxial" in stem.lower():
            stress = df["s_zz"].values / 1e6
            ylabel = r"$\sigma_{zz}$ (MPa)"
        else:
            stress = df["s_xy"].values / 1e6
            ylabel = r"$\tau_{xy}$ (MPa)"

        ep_true = df["ep_eq_true"].values * 100.0
        ep_pred = df["ep_eq_pred"].values * 100.0
        ax.plot(ep_true, stress, label="DDD true path", linewidth=2)
        ax.plot(ep_pred, stress, "--", label="LSTM rollout path", linewidth=2)
        ax.set_title(stem, fontsize=9)
        ax.set_xlabel(r"$\varepsilon_{eq}^{p}$ (%)")
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)

    for j in range(len(csv_files), len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()


def plot_rollout_density_vs_plastic_strain(pattern_prefix, out_dir, max_plots=None):
    import glob
    csv_files = sorted(glob.glob(os.path.join(out_dir, f"{pattern_prefix}_*.csv")))
    csv_files = [f for f in csv_files if "summary" not in os.path.basename(f).lower()]
    if not csv_files:
        print(f"No rollout CSV files found for {pattern_prefix}.")
        return
    if max_plots is not None:
        csv_files = csv_files[:max_plots]

    n = len(csv_files)
    ncols = 2
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4.5 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for ax, fpath in zip(axes, csv_files):
        df = pd.read_csv(fpath)
        stem = str(df["stem"].iloc[0])
        ep_true = df["ep_eq_true"].values * 100.0
        ax.plot(ep_true, df["rho_true"].values, label="DDD true density", linewidth=2)
        ax.plot(ep_true, df["rho_pred"].values, "--", label="LSTM rollout density", linewidth=2)
        ax.set_title(stem, fontsize=9)
        ax.set_xlabel(r"$\varepsilon_{eq}^{p}$ (%)")
        ax.set_ylabel(r"$\rho$ (m$^{-2}$)")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)

    for j in range(len(csv_files), len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()



# MAIN FUNCTION

def train_and_evaluate_lstm(stats_path, seq_path):
    print(f"Device : {DEVICE}")
    print(f"PyTorch: {torch.__version__}\n")

    stats = json.load(open(stats_path))

    dp_train_ds = DpSequenceDataset(seq_path, stats, split="train", window_len=WINDOW_LEN, stride=WINDOW_STRIDE,
                                    max_windows_per_run=MAX_WINDOWS_PER_RUN)
    dp_val_ds = DpSequenceDataset(seq_path, stats, split="val", window_len=WINDOW_LEN, stride=WINDOW_STRIDE,
                                  max_windows_per_run=MAX_WINDOWS_PER_RUN)
    dp_test_ds = DpSequenceDataset(seq_path, stats, split="test", window_len=WINDOW_LEN, stride=WINDOW_STRIDE,
                                   max_windows_per_run=MAX_WINDOWS_PER_RUN)

    rho_train_ds = RhoSequenceDataset(seq_path, stats, split="train", window_len=WINDOW_LEN, stride=WINDOW_STRIDE,
                                      max_windows_per_run=MAX_WINDOWS_PER_RUN)
    rho_val_ds = RhoSequenceDataset(seq_path, stats, split="val", window_len=WINDOW_LEN, stride=WINDOW_STRIDE,
                                    max_windows_per_run=MAX_WINDOWS_PER_RUN)
    rho_test_ds = RhoSequenceDataset(seq_path, stats, split="test", window_len=WINDOW_LEN, stride=WINDOW_STRIDE,
                                     max_windows_per_run=MAX_WINDOWS_PER_RUN)

    dp_train_loader = DataLoader(dp_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    dp_val_loader = DataLoader(dp_val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    dp_test_loader = DataLoader(dp_test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    rho_train_loader = DataLoader(rho_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    rho_val_loader = DataLoader(rho_val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    rho_test_loader = DataLoader(rho_test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    print(f"Dp windows: train={len(dp_train_ds):,}, val={len(dp_val_ds):,}, test={len(dp_test_ds):,}")
    print(f"Rho windows: train={len(rho_train_ds):,}, val={len(rho_val_ds):,}, test={len(rho_test_ds):,}")

    dp_model = LSTMRegressor(
        input_dim=DP_INPUT_DIM,
        hidden_size=DP_HIDDEN_SIZE,
        num_layers=DP_NUM_LAYERS,
        output_dim=5,
        head_hidden=DP_HEAD_HIDDEN,
        dropout=DP_DROPOUT,
    ).to(DEVICE)

    rho_model = LSTMRegressor(
        input_dim=RHO_INPUT_DIM,
        hidden_size=RHO_HIDDEN_SIZE,
        num_layers=RHO_NUM_LAYERS,
        output_dim=1,
        head_hidden=RHO_HEAD_HIDDEN,
        dropout=RHO_DROPOUT,
    ).to(DEVICE)

    print(f"\nDp-LSTM params: {sum(p.numel() for p in dp_model.parameters() if p.requires_grad):,}")
    print(dp_model)
    print(f"\nRho-LSTM params: {sum(p.numel() for p in rho_model.parameters() if p.requires_grad):,}")
    print(rho_model)

    dp_weights_path = os.path.join(NN_OUT_DIR, "dp_lstm_best_state.pt")
    rho_weights_path = os.path.join(NN_OUT_DIR, "rho_lstm_best_state.pt")

    dp_model, dp_history, dp_best_val = train_dp_lstm(dp_model, dp_train_loader, dp_val_loader, stats, dp_weights_path)
    rho_model, rho_history, rho_best_val = train_rho_lstm(rho_model, rho_train_loader, rho_val_loader, stats, rho_weights_path)

    torch.save({
        "model_state_dict": dp_model.state_dict(),
        "model_type": "LSTM",
        "input_dim": DP_INPUT_DIM,
        "hidden_size": DP_HIDDEN_SIZE,
        "num_layers": DP_NUM_LAYERS,
        "output_dim": 5,
        "stats": stats,
        "best_val_loss": dp_best_val,
    }, os.path.join(NN_OUT_DIR, "dp_lstm_best.pt"))

    torch.save({
        "model_state_dict": rho_model.state_dict(),
        "model_type": "LSTM",
        "input_dim": RHO_INPUT_DIM,
        "hidden_size": RHO_HIDDEN_SIZE,
        "num_layers": RHO_NUM_LAYERS,
        "output_dim": 1,
        "stats": stats,
        "best_val_loss": rho_best_val,
    }, os.path.join(NN_OUT_DIR, "rho_lstm_best.pt"))

    print("\nEvaluating Dp-LSTM on train/val/test ...")
    dp_r2s, dp_rmses = {}, {}
    for split, loader in [("train", dp_train_loader), ("val", dp_val_loader), ("test", dp_test_loader)]:
        r2, rmse, _, _ = evaluate_dp_lstm(dp_model, loader, stats, DEVICE)
        dp_r2s[split] = r2
        dp_rmses[split] = rmse

    print(f"\n{'Component':<12}{'R²_train':>10}{'R²_val':>10}{'R²_test':>10}{'RMSE_test':>12}")
    print("-" * 56)
    for i, label in enumerate(DP_OUTPUT_LABELS_6):
        note = " *" if label == "Dp_zz" else "  "
        print(f"  {label:<8}{note}  {dp_r2s['train'][i]:>8.4f}  {dp_r2s['val'][i]:>8.4f}  {dp_r2s['test'][i]:>8.4f}  {dp_rmses['test'][i]:>10.2f}")
    print("-" * 56)
    print(f"  {'MEAN':<10}  {dp_r2s['train'].mean():>8.4f}  {dp_r2s['val'].mean():>8.4f}  {dp_r2s['test'].mean():>8.4f}  {dp_rmses['test'].mean():>10.2f}")

    print("\nEvaluating Rho-LSTM on train/val/test ...")
    rho_metrics = {}
    for split, loader in [("train", rho_train_loader), ("val", rho_val_loader), ("test", rho_test_loader)]:
        r2, rmse, _, _ = evaluate_rho_lstm(rho_model, loader, stats, DEVICE)
        rho_metrics[split] = dict(r2=r2, rmse=rmse)

    print(f"\n{'Split':<10}{'R²':>12}{'RMSE':>16}")
    print("-" * 38)
    for split in ["train", "val", "test"]:
        print(f"{split:<10}{rho_metrics[split]['r2']:>12.4f}{rho_metrics[split]['rmse']:>16.4e}")

    print("\nGenerating plots ...")
    plot_training_curves(dp_history, os.path.join(NN_OUT_DIR, "dp_lstm"))
    plot_training_curves(rho_history, os.path.join(NN_OUT_DIR, "rho_lstm"))
    plot_branch_training_detail(dp_history, os.path.join(NN_OUT_DIR, "dp_lstm_training_detail.png"),
                                "Dp-LSTM step vs rollout losses")
    plot_branch_training_detail(rho_history, os.path.join(NN_OUT_DIR, "rho_lstm_training_detail.png"),
                                "Rho-LSTM step vs rollout losses")
    plot_dp_per_component(dp_r2s, dp_rmses, NN_OUT_DIR)

    rollout_summary_A = rollout_all_test_runs_modeA_lstm(dp_model, seq_path, stats, NN_OUT_DIR)
    rollout_summary_B = rollout_all_test_runs_modeB_lstm(dp_model, rho_model, seq_path, stats, NN_OUT_DIR)

    print("\nDisplaying rollout Mode A: stress vs plastic strain ...")
    plot_rollout_stress_vs_plastic_strain("rollout_modeA", NN_OUT_DIR)

    print("\nDisplaying rollout Mode B: stress vs plastic strain ...")
    plot_rollout_stress_vs_plastic_strain("rollout_modeB", NN_OUT_DIR)

    print("\nDisplaying rollout Mode B: density vs plastic strain ...")
    plot_rollout_density_vs_plastic_strain("rollout_modeB", NN_OUT_DIR)

    print(f"\nAll outputs in: {NN_OUT_DIR}")
    return dp_model, rho_model, dp_history, rho_history, dp_r2s, rho_metrics, rollout_summary_A, rollout_summary_B


if __name__ == "__main__":
    set_all_seeds(RANDOM_SEED)
    dataset_paths = build_dataset_lstm()
    train_and_evaluate_lstm(
        stats_path=dataset_paths["stats_path"],
        seq_path=dataset_paths["seq_path"],
    )
